In [ ]:
# In[1]:

import pandas as pd
import pytz
from datetime import datetime

# timezone UTC+8
tz = pytz.timezone('Asia/Shanghai')

# Load CSVs (files are in current working directory)
metric_df = pd.read_csv('metric.csv')
trace_df = pd.read_csv('trace.csv')
log_df = pd.read_csv('log.csv')
error_logs_df = pd.read_csv('error_logs.csv')

# helper to convert epoch seconds to timezone-aware string in UTC+8
def ts_range_info(series):
    if series.empty or series.dropna().shape[0] == 0:
        return (None, None)
    mn = int(series.min())
    mx = int(series.max())
    mn_s = pd.to_datetime(mn, unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z%z')
    mx_s = pd.to_datetime(mx, unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z%z')
    return mn_s, mx_s

# helper to get sorted distinct samples limited to n
def sorted_distinct_limited(series, limit):
    if series is None or len(series) == 0:
        return []
    vals = sorted(pd.Series(series).dropna().unique().astype(str).tolist())
    return vals[:limit]

# Summarize metric.csv
metric_rows = len(metric_df)
metric_min_ts, metric_max_ts = ts_range_info(metric_df['timestamp']) if 'timestamp' in metric_df.columns else (None, None)
metric_cmdbs = sorted_distinct_limited(metric_df['cmdb_id'] if 'cmdb_id' in metric_df.columns else [], 20)
metric_kpis = sorted_distinct_limited(metric_df['kpi_name'] if 'kpi_name' in metric_df.columns else [], 50)
metric_summary = pd.DataFrame([{
    'file': 'metric.csv',
    'rows': metric_rows,
    'min_timestamp_UTC+8': metric_min_ts,
    'max_timestamp_UTC+8': metric_max_ts,
    'distinct_cmdb_count': len(pd.Series(metric_cmdbs)),
    'cmdb_sample_top20': metric_cmdbs,
    'distinct_kpi_count': len(metric_kpis),
    'kpi_sample_top50': metric_kpis
}])

# Summarize trace.csv
trace_rows = len(trace_df)
trace_min_ts, trace_max_ts = ts_range_info(trace_df['timestamp']) if 'timestamp' in trace_df.columns else (None, None)
trace_cmdbs = sorted_distinct_limited(trace_df['cmdb_id'] if 'cmdb_id' in trace_df.columns else [], 20)
trace_names = sorted_distinct_limited(trace_df['trace_name'] if 'trace_name' in trace_df.columns else [], 50)
trace_summary = pd.DataFrame([{
    'file': 'trace.csv',
    'rows': trace_rows,
    'min_timestamp_UTC+8': trace_min_ts,
    'max_timestamp_UTC+8': trace_max_ts,
    'distinct_cmdb_count': len(trace_cmdbs),
    'cmdb_sample_top20': trace_cmdbs,
    'distinct_trace_count': len(trace_names),
    'trace_sample_top50': trace_names
}])

# Summarize log.csv
log_rows = len(log_df)
log_min_ts, log_max_ts = ts_range_info(log_df['timestamp']) if 'timestamp' in log_df.columns else (None, None)
log_cmdbs = sorted_distinct_limited(log_df['cmdb_id'] if 'cmdb_id' in log_df.columns else [], 20)
log_names = sorted_distinct_limited(log_df['log_name'] if 'log_name' in log_df.columns else [], 50)
log_summary = pd.DataFrame([{
    'file': 'log.csv',
    'rows': log_rows,
    'min_timestamp_UTC+8': log_min_ts,
    'max_timestamp_UTC+8': log_max_ts,
    'distinct_cmdb_count': len(log_cmdbs),
    'cmdb_sample_top20': log_cmdbs,
    'distinct_log_count': len(log_names),
    'log_sample_top50': log_names
}])

# Summarize error_logs.csv (include message samples up to 10)
err_rows = len(error_logs_df)
err_min_ts, err_max_ts = ts_range_info(error_logs_df['timestamp']) if 'timestamp' in error_logs_df.columns else (None, None)
err_cmdbs = sorted_distinct_limited(error_logs_df['cmdb_id'] if 'cmdb_id' in error_logs_df.columns else [], 20)
err_messages = sorted_distinct_limited(error_logs_df['message'] if 'message' in error_logs_df.columns else [], 10)
error_logs_summary = pd.DataFrame([{
    'file': 'error_logs.csv',
    'rows': err_rows,
    'min_timestamp_UTC+8': err_min_ts,
    'max_timestamp_UTC+8': err_max_ts,
    'distinct_cmdb_count': len(err_cmdbs),
    'cmdb_sample_top20': err_cmdbs,
    'distinct_message_count': len(pd.Series(error_logs_df['message']).dropna().unique()) if 'message' in error_logs_df.columns else 0,
    'message_samples_top10': err_messages
}])

# Display compact summaries (use variables, not print)
metric_summary, trace_summary, log_summary, error_logs_summary

```
Out[1]:
```
summary = (
    "Summary of loaded telemetry files (timestamps shown in UTC+8):\n\n"
    "metric.csv:\n"
    "- rows: 319,507\n"
    "- time range: 2022-03-20 08:00:00 to 2022-03-20 09:00:00\n"
    "- distinct cmdb_id count: 20 (sample includes adservice, adservice-0, adservice-1, ...)\n"
    "- distinct kpi count: 50 (many KPI names; samples include app.grpc.count, app.grpc.mrt, ...)\n\n"
    "trace.csv:\n"
    "- rows: 31,120\n"
    "- time range: 2022-03-20 08:30:00 to 2022-03-20 08:59:00\n"
    "- distinct cmdb_id count: 20 (sample includes adservice-0, adservice-1, ...)\n"
    "- distinct trace count: 50 (many trace names; samples include trace.from_checkoutservice-0.duration_mean, ...)\n\n"
    "log.csv:\n"
    "- rows: 1,808\n"
    "- time range: 2022-03-20 08:30:00 to 2022-03-20 09:00:00\n"
    "- distinct cmdb_id count: 20 (sample includes adservice-0, adservice-1, cartservice-0, ...)\n"
    "- distinct log count: 2 (log.error_count, log.row_count)\n\n"
    "error_logs.csv:\n"
    "- rows: 199\n"
    "- time range: 2022-03-20 08:48:52 to 2022-03-20 08:51:53\n"
    "- distinct cmdb_id count: 3 (frontend-0, frontend-1, frontend-2)\n"
    "- distinct message count: 1\n"
    "- message samples (up to 10): ['severity: warning, message: failed to retriev...']\n\n"
    "Notes: outputs are compact summaries only; no anomaly detection performed yet."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(         file    rows           min_timestamp_UTC+8           max_timestamp_UTC+8  distinct_cmdb_count                                  cmdb_sample_top20  distinct_kpi_count                                   kpi_sample_top50
0  metric.csv  319507  2022-03-20 08:00:00 CST+0800  2022-03-20 09:00:00 CST+0800                   20  [adservice, adservice-0, adservice-1, adservic...                  50  [app.grpc.count, app.grpc.mrt, app.grpc.rr, ap...,         file   rows           min_timestamp_UTC+8           max_timestamp_UTC+8  distinct_cmdb_count                                  cmdb_sample_top20  distinct_trace_count                                 trace_sample_top50
0  trace.csv  31120  2022-03-20 08:30:00 CST+0800  2022-03-20 08:59:00 CST+0800                   20  [adservice-0, adservice-1, adservice-2, adserv...                    50  [trace.from_checkoutservice-0.duration_mean, t...,       file  rows           min_timestamp_UTC+8           max_timestamp_UTC+8  distinct_cmdb_count                                  cmdb_sample_top20  distinct_log_count                  log_sample_top50
0  log.csv  1808  2022-03-20 08:30:00 CST+0800  2022-03-20 09:00:00 CST+0800                   20  [adservice-0, adservice-1, adservice-2, cartse...                   2  [log.error_count, log.row_count],              file  rows           min_timestamp_UTC+8           max_timestamp_UTC+8  distinct_cmdb_count                     cmdb_sample_top20  distinct_message_count                              message_samples_top10
0  error_logs.csv   199  2022-03-20 08:48:52 CST+0800  2022-03-20 08:51:53 CST+0800                    3  [frontend-0, frontend-1, frontend-2]                       1  [severity: warning, message: failed to retriev...)```
```

In [ ]:
# In[2]:

# Reuse existing metric_df (loaded previously). Compute global statistics per (cmdb_id, kpi_name).
import pandas as pd

# Ensure numeric values
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# Aggregate global statistics across entire series for each (cmdb_id, kpi_name)
metric_thresholds = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'], observed=True)['value']
    .agg(
        count=lambda x: x.count(),
        P50=lambda x: x.quantile(0.50),
        P75=lambda x: x.quantile(0.75),
        P90=lambda x: x.quantile(0.90),
        P95=lambda x: x.quantile(0.95),
        mean='mean',
        max='max'
    )
    .reset_index()
)

# Compact summary: total unique pairs and top 20 rows ordered by descending P95
total_pairs = metric_thresholds.shape[0]
top20_by_P95 = metric_thresholds.sort_values('P95', ascending=False)[
    ['cmdb_id', 'kpi_name', 'count', 'P95', 'P90', 'P75', 'mean', 'max']
].head(20)

# Display compact results (metric_thresholds is stored in kernel as requested)
total_pairs, top20_by_P95

```
Out[2]:
```
summary_lines = []
total_pairs = int(metric_thresholds.shape[0])
summary_lines.append(f"Computed global statistics (count, P50, P75, P90, P95, mean, max) for each (cmdb_id, kpi_name) using the entire metric.csv.")
summary_lines.append(f"Total unique (cmdb_id, kpi_name) pairs: {total_pairs}. No time filtering or anomaly detection applied.")
summary_lines.append("")
summary_lines.append("Top 20 (cmdb_id, kpi_name) by descending P95 — showing cmdb_id, kpi_name, count, P95:")
top20 = (
    metric_thresholds
    .sort_values('P95', ascending=False)
    [['cmdb_id', 'kpi_name', 'count', 'P95']]
    .head(20)
    .reset_index(drop=True)
)
# Format rows compactly
for i, row in top20.iterrows():
    p95_fmt = f"{row['P95']:.6g}"
    summary_lines.append(f"{i+1:2d}. {row['cmdb_id']} | {row['kpi_name']} | count={int(row['count'])} | P95={p95_fmt}")

summary_lines.append("")
summary_lines.append("Note: Full result table is stored in kernel variable 'metric_thresholds'.")
summary = "\n".join(summary_lines)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(5253,          cmdb_id                                           kpi_name  count           P95           P90           P75          mean           max
244    adservice   runtime.java_lang_OperatingSystem_ProcessCpuTime     61  1.905145e+12  1.904637e+12  1.903122e+12  1.900619e+12  1.905646e+12
857   adservice2                runtime.java_lang_Runtime_StartTime     61  1.647263e+12  1.647263e+12  1.647263e+12  1.647263e+12  1.647263e+12
250    adservice                runtime.java_lang_Runtime_StartTime     61  1.647144e+12  1.647144e+12  1.647144e+12  1.647144e+12  1.647144e+12
851   adservice2   runtime.java_lang_OperatingSystem_ProcessCpuTime     61  1.523845e+12  1.523381e+12  1.521998e+12  1.519695e+12  1.524306e+12
253    adservice   runtime.java_lang_Threading_CurrentThreadCpuTime     61  1.989167e+11  1.988492e+11  1.986605e+11  1.983677e+11  1.989795e+11
255    adservice  runtime.java_lang_Threading_CurrentThreadUserTime     61  1.812867e+11  1.812067e+11  1.810325e+11  1.807621e+11  1.813783e+11
860   adservice2   runtime.java_lang_Threading_CurrentThreadCpuTime     61  1.600967e+11  1.600080e+11  1.598690e+11  1.595497e+11  1.601702e+11
862   adservice2  runtime.java_lang_Threading_CurrentThreadUserTime     61  1.501592e+11  1.501033e+11  1.499408e+11  1.496566e+11  1.502200e+11
252    adservice  runtime.java_lang_Threading_CurrentThreadAlloc...     61  1.438827e+11  1.438393e+11  1.437079e+11  1.434904e+11  1.439261e+11
859   adservice2  runtime.java_lang_Threading_CurrentThreadAlloc...     61  1.161204e+11  1.160752e+11  1.159448e+11  1.157240e+11  1.161648e+11
3643      node-5                                  system.disk.total     61  1.854945e+10  1.854945e+10  1.854945e+10  1.854945e+10  1.854945e+10
3647      node-5                             system.fs.inodes.total     61  1.563454e+10  1.563330e+10  1.563028e+10  1.562769e+10  1.563522e+10
3645      node-5                              system.fs.inodes.free     61  1.562993e+10  1.562868e+10  1.562566e+10  1.562308e+10  1.563061e+10
3644      node-5                                   system.disk.used     61  1.453738e+10  1.453729e+10  1.453651e+10  1.453577e+10  1.453869e+10
3526      node-3                                  system.disk.total     61  1.406688e+10  1.406688e+10  1.406688e+10  1.406688e+10  1.406688e+10
3468      node-2                                  system.disk.total     61  1.406688e+10  1.406688e+10  1.406688e+10  1.406688e+10  1.406688e+10
3409      node-1                                  system.disk.total     61  1.406688e+10  1.406688e+10  1.406688e+10  1.406688e+10  1.406688e+10
3585      node-4                                  system.disk.total     45  1.406688e+10  1.406688e+10  1.406688e+10  1.406688e+10  1.406688e+10
3413      node-1                             system.fs.inodes.total     61  1.173575e+10  1.173530e+10  1.173267e+10  1.173066e+10  1.173639e+10
3530      node-3                             system.fs.inodes.total     61  1.173574e+10  1.173534e+10  1.173273e+10  1.173067e+10  1.173649e+10)```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz
from datetime import datetime

# reuse metric_df and metric_thresholds from kernel
# ensure tz for UTC+8
tz = pytz.timezone('Asia/Shanghai')

# define window in UTC+8 then convert to UTC epoch seconds
start_local = tz.localize(datetime(2022, 3, 20, 8, 30, 0))
end_local = tz.localize(datetime(2022, 3, 20, 9, 0, 0))
start_epoch = int(start_local.astimezone(pytz.UTC).timestamp())
end_epoch = int(end_local.astimezone(pytz.UTC).timestamp())

# Filter metric_df to window and store as metric_window
metric_window = metric_df[(metric_df['timestamp'] >= start_epoch) & (metric_df['timestamp'] <= end_epoch)].copy()

# Join P95 from metric_thresholds (which was computed globally)
# Ensure P95 exists in metric_thresholds
metric_window = metric_window.merge(
    metric_thresholds[['cmdb_id', 'kpi_name', 'P95']],
    on=['cmdb_id', 'kpi_name'],
    how='left'
)

# Mark anomalies: value >= P95 (if P95 is NaN -> not anomaly)
metric_window['anomaly'] = np.where(
    metric_window['P95'].notna() & (metric_window['value'] >= metric_window['P95']),
    True,
    False
)

# Store anomaly rows in metric_anomalies
metric_anomalies = metric_window[metric_window['anomaly']].copy()

# Aggregations per (cmdb_id, kpi_name)
grp = metric_window.groupby(['cmdb_id', 'kpi_name'], observed=True)

agg = grp.agg(
    total_points_in_window=('value', 'count'),
    anomaly_count=('anomaly', 'sum'),
    earliest_anom_ts_epoch=('timestamp', lambda s: s[ metric_window.loc[s.index, 'anomaly'] ].min() if (metric_window.loc[s.index, 'anomaly'].any()) else pd.NA),
    latest_anom_ts_epoch=('timestamp', lambda s: s[ metric_window.loc[s.index, 'anomaly'] ].max() if (metric_window.loc[s.index, 'anomaly'].any()) else pd.NA),
    max_value_in_window=('value', 'max'),
    P95=('P95', 'first')
).reset_index()

# convert epoch timestamps to UTC+8 readable strings, keep compact
def fmt_epoch(e):
    if pd.isna(e):
        return pd.NA
    return pd.to_datetime(int(e), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z%z')

agg['earliest_anom_timestamp'] = agg['earliest_anom_ts_epoch'].apply(fmt_epoch)
agg['latest_anom_timestamp'] = agg['latest_anom_ts_epoch'].apply(fmt_epoch)

# compute severity, handle P95 == 0 or NaN -> set NaN
agg['severity'] = np.where(
    agg['P95'].notna() & (agg['P95'] != 0),
    (agg['max_value_in_window'] - agg['P95']) / agg['P95'],
    np.nan
)

# Prepare the two compact result tables
top_by_anomaly_count = (
    agg
    .sort_values(['anomaly_count', 'total_points_in_window'], ascending=[False, False])
    [['cmdb_id', 'kpi_name', 'total_points_in_window', 'anomaly_count', 'earliest_anom_timestamp', 'latest_anom_timestamp', 'P95']]
    .head(20)
    .reset_index(drop=True)
)

top_by_severity = (
    agg
    .sort_values('severity', ascending=False)
    [['cmdb_id', 'kpi_name', 'anomaly_count', 'max_value_in_window', 'P95', 'severity']]
    .head(20)
    .reset_index(drop=True)
)

# Display compact summaries (metric_window and metric_anomalies are stored in kernel variables as requested)
top_by_anomaly_count, top_by_severity

```
Out[3]:
```
summary = (
    "Summary of metric-window anomaly analysis (window: 2022-03-20 08:30:00 to 2022-03-20 09:00:00 UTC+8):\n\n"
    "What I did:\n"
    "- Filtered metric.csv to the specified window and joined global P95 (from metric_thresholds).\n"
    "- Marked points as anomalies when value >= P95.\n"
    "- Stored the filtered series in kernel variable 'metric_window' and anomaly rows in 'metric_anomalies'.\n\n"
    "Key findings (compact):\n"
    "1) By anomaly count (top results):\n"
    "- Many adservice KPIs (e.g., app.grpc.rr, app.http.count, app.http.rr, app.http.sr, many runtime.* and GC.* metrics) show anomaly_count = 31 of 31 points — i.e., every point in the window >= P95.\n"
    "- For these adservice rows the earliest anomalous timestamp is 2022-03-20 08:30:00 and the latest is 2022-03-20 09:00:00 (UTC+8), indicating sustained exceedance across the whole window.\n\n"
    "2) By severity (top results):\n"
    "- The largest severity ratios are for mesh/istio metrics (high relative spikes):\n"
    "  * checkoutservice-1 mesh.source...  (anomaly_count=4) max=4.1625 vs P95=0.01025 => severity ≈ 405x\n"
    "  * checkoutservice2-0 mesh...         (anomaly_count=3) max=1.65 vs P95=0.0041 => severity ≈ 401x\n"
    "  * cartservice2-0 mesh.destination... (anomaly_count=4) max=0.415 vs P95=0.0021 => severity ≈ 197x\n"
    "- Other high-severity items include frontend2-0 mesh metrics, node-1 TCP retransmissions, node-4 write I/O (system.io.w_s), and several container network receive MB metrics for payment/email/shipping services (severity ~16–22x).\n\n"
    "Interpretation / next steps (recommended):\n"
    "- There are two notable patterns: (1) many adservice KPIs are continuously above their global P95 in the window (sustained elevation), and (2) several mesh/network-related metrics show very large relative spikes (high severity ratios).\n"
    "- These point to potential networking/mesh traffic anomalies and sustained elevated behavior in adservice. Next steps: inspect traces and logs for the high-severity mesh metrics and the adservice pods to identify root cause.\n\n"
    "Notes:\n"
    "- The full per-(cmdb_id,kpi_name) aggregated results are in kernel variable 'metric_thresholds'.\n"
    "- The filtered time-series and anomaly rows are stored in 'metric_window' and 'metric_anomalies', respectively.\n"
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id                                           kpi_name  total_points_in_window  anomaly_count       earliest_anom_timestamp         latest_anom_timestamp          P95
0   adservice                                        app.grpc.rr                      31             31  2022-03-20 08:30:00 CST+0800  2022-03-20 09:00:00 CST+0800        100.0
1   adservice                                     app.http.count                      31             31  2022-03-20 08:30:00 CST+0800  2022-03-20 09:00:00 CST+0800         24.0
2   adservice                                        app.http.rr                      31             31  2022-03-20 08:30:00 CST+0800  2022-03-20 09:00:00 CST+0800        100.0
3   adservice                                        app.http.sr                      31             31  2022-03-20 08:30:00 CST+0800  2022-03-20 09:00:00 CST+0800        100.0
4   adservice    runtime.java_lang_ClassLoading_LoadedClassCount                      31             31  2022-03-20 08:30:00 CST+0800  2022-03-20 09:00:00 CST+0800       5162.0
5   adservice  runtime.java_lang_ClassLoading_TotalLoadedClas...                      31             31  2022-03-20 08:30:00 CST+0800  2022-03-20 09:00:00 CST+0800       5185.0
6   adservice  runtime.java_lang_ClassLoading_UnloadedClassCount                      31             31  2022-03-20 08:30:00 CST+0800  2022-03-20 09:00:00 CST+0800         23.0
7   adservice             runtime.java_lang_ClassLoading_Verbose                      31             31  2022-03-20 08:30:00 CST+0800  2022-03-20 09:00:00 CST+0800          0.0
8   adservice  runtime.java_lang_Compilation_CompilationTimeM...                      31             31  2022-03-20 08:30:00 CST+0800  2022-03-20 09:00:00 CST+0800          1.0
9   adservice  runtime.java_lang_Compilation_TotalCompilation...                      31             31  2022-03-20 08:30:00 CST+0800  2022-03-20 09:00:00 CST+0800      47013.0
10  adservice  runtime.java_lang_GarbageCollector_CollectionC...                      31             31  2022-03-20 08:30:00 CST+0800  2022-03-20 09:00:00 CST+0800          8.0
11  adservice  runtime.java_lang_GarbageCollector_CollectionT...                      31             31  2022-03-20 08:30:00 CST+0800  2022-03-20 09:00:00 CST+0800        480.0
12  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...                      31             31  2022-03-20 08:30:00 CST+0800  2022-03-20 09:00:00 CST+0800          1.0
13  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...                      31             31  2022-03-20 08:30:00 CST+0800  2022-03-20 09:00:00 CST+0800          1.0
14  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...                      31             31  2022-03-20 08:30:00 CST+0800  2022-03-20 09:00:00 CST+0800         94.0
15  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...                      31             31  2022-03-20 08:30:00 CST+0800  2022-03-20 09:00:00 CST+0800  539245384.0
16  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...                      31             31  2022-03-20 08:30:00 CST+0800  2022-03-20 09:00:00 CST+0800          8.0
17  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...                      31             31  2022-03-20 08:30:00 CST+0800  2022-03-20 09:00:00 CST+0800   23330816.0
18  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...                      31             31  2022-03-20 08:30:00 CST+0800  2022-03-20 09:00:00 CST+0800   23330816.0
19  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...                      31             31  2022-03-20 08:30:00 CST+0800  2022-03-20 09:00:00 CST+0800    3932160.0,                cmdb_id                                           kpi_name  anomaly_count  max_value_in_window       P95    severity
0    checkoutservice-1  mesh.source.checkoutservice.cartservice.istio_...              4             4.162500  0.010250  405.097561
1   checkoutservice2-0  mesh.source.checkoutservice2.cartservice2.isti...              3             1.650000  0.004100  401.439024
2       cartservice2-0  mesh.destination.checkoutservice2.cartservice2...              4             0.415000  0.002100  196.619048
3          frontend2-0  mesh.destination.unknown.frontend2.istio_reque...              5             1.875000  0.024000   77.125000
4    checkoutservice-2  mesh.source.checkoutservice.cartservice.istio_...              3             2.475000  0.044650   54.431131
5        cartservice-1  mesh.destination.checkoutservice.cartservice.i...              3             1.453025  0.039600   35.692551
6               node-1                        system.net.tcp.retrans_segs              2             6.050000  0.180000   32.611111
7               node-4                                      system.io.w_s              4            15.000000  0.500000   29.000000
8     paymentservice-0  container.node-6.container_network_receive_MB....              1             0.593897  0.025768   22.047594
9      emailservice2-0  container.node-6.container_network_receive_MB....              2             0.618471  0.027747   21.289517
10    paymentservice-2  container.node-6.container_network_receive_MB....              2             0.606353  0.027423   21.111198
11          frontend-1  mesh.source.frontend.checkoutservice.istio_req...              3             3.950000  0.206750   18.105200
12   checkoutservice-1  mesh.destination.frontend.checkoutservice.isti...              4             3.575000  0.187750   18.041278
13  checkoutservice2-0  mesh.destination.frontend2.checkoutservice2.is...              3             1.650000  0.089500   17.435754
14         frontend2-0  mesh.source.frontend2.checkoutservice2.istio_r...              3             1.650000  0.090500   17.232044
15          frontend-2  mesh.source.frontend.checkoutservice.istio_req...              4             2.975000  0.165250   17.003026
16       redis-cart2-0  container.node-6.container_network_receive_MB....              2             0.608264  0.035158   16.301040
17   shippingservice-1  container.node-6.container_network_receive_MB....              2             0.607366  0.035328   16.192012
18     checkoutservice                                       app.grpc.mrt              3             2.060000  0.121000   16.024793
19   shippingservice-2  container.node-5.container_network_receive_MB....              3             0.610693  0.036114   15.910280)```
```

In [ ]:
# In[4]:

import pandas as pd

# reuse trace_df from kernel, ensure numeric values
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

# compute global statistics per (cmdb_id, trace_name)
trace_thresholds = (
    trace_df
    .groupby(['cmdb_id', 'trace_name'], observed=True)['value']
    .agg(
        count=lambda x: x.count(),
        P50=lambda x: x.quantile(0.50),
        P75=lambda x: x.quantile(0.75),
        P90=lambda x: x.quantile(0.90),
        P95=lambda x: x.quantile(0.95),
        mean='mean',
        max='max'
    )
    .reset_index()
)

# compact summary outputs
total_pairs = trace_thresholds.shape[0]
top20_by_P95 = trace_thresholds.sort_values('P95', ascending=False)[
    ['cmdb_id', 'trace_name', 'count', 'P95', 'P90', 'P75', 'mean', 'max']
].head(20)

# Display compact summary (trace_thresholds stored in kernel)
total_pairs, top20_by_P95

```
Out[4]:
```
summary = (
    "Summary of global trace statistics (computed from full trace.csv):\n\n"
    "1) Total unique (cmdb_id, trace_name) pairs: 1,232.\n\n"
    "2) Top traces by global P95 (top 20, compact highlights):\n"
    "- frontend-2 | trace.self.row_count — count=30, P95=454.20, P90=450.2, P75=390.25, mean=374.37, max=478.0\n"
    "- frontend-0 | trace.self.row_count — count=30, P95=432.65, P90=410.1, P75=396.25, mean=370.37, max=499.0\n"
    "- frontend-1 | trace.self.row_count — count=30, P95=432.55, P90=426.6, P75=412.50, mean=381.60, max=443.0\n"
    "- frontend2-0 | trace.self.row_count — count=30, P95=169.05, P90=160.3, P75=128.25, mean=120.27, max=194.0\n"
    "- Several productcatalogservice and frontend-to-productcatalogservice row_count traces follow (P95 ~70–80, counts=30), indicating higher row counts on frontend and productcatalogservice interactions.\n\n"
    "Interpretation / note:\n"
    "- The highest P95 values are from frontend self row_count metrics (sustained high row counts across 30 samples).\n"
    "- Many of the top traces reflect frontend and frontend→productcatalogservice interactions, suggesting heavy request/row activity there.\n"
    "- The full per-(cmdb_id,trace_name) table is stored in kernel variable 'trace_thresholds' for further analysis or anomaly detection.\n"
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(1232,                        cmdb_id                                   trace_name  count     P95    P90     P75        mean    max
759                 frontend-2                         trace.self.row_count     30  454.20  450.2  390.25  374.366667  478.0
575                 frontend-0                         trace.self.row_count     30  432.65  410.1  396.25  370.366667  499.0
667                 frontend-1                         trace.self.row_count     30  432.55  426.6  412.50  381.600000  443.0
851                frontend2-0                         trace.self.row_count     30  169.05  160.3  128.25  120.266667  194.0
1047  productcatalogservice2-0             trace.from_frontend2-0.row_count     30   80.40   74.2   65.50   55.833333   87.0
871                frontend2-0  trace.to_productcatalogservice2-0.row_count     30   80.40   74.2   65.50   55.833333   87.0
819                 frontend-2   trace.to_productcatalogservice-2.row_count     30   70.55   68.2   61.75   57.833333   71.0
1023   productcatalogservice-2              trace.from_frontend-2.row_count     30   70.55   68.2   61.75   57.833333   71.0
943    productcatalogservice-0              trace.from_frontend-2.row_count     30   70.10   68.1   61.75   57.833333   71.0
811                 frontend-2   trace.to_productcatalogservice-0.row_count     30   70.10   68.1   61.75   57.833333   71.0
719                 frontend-1   trace.to_productcatalogservice-0.row_count     30   69.65   65.3   63.25   58.700000   71.0
939    productcatalogservice-0              trace.from_frontend-1.row_count     30   69.65   65.3   63.25   58.700000   71.0
815                 frontend-2   trace.to_productcatalogservice-1.row_count     30   69.55   68.1   61.75   57.833333   71.0
983    productcatalogservice-1              trace.from_frontend-2.row_count     30   69.55   68.1   61.75   57.833333   71.0
1019   productcatalogservice-2              trace.from_frontend-1.row_count     30   69.10   65.3   63.25   58.700000   71.0
727                 frontend-1   trace.to_productcatalogservice-2.row_count     30   69.10   65.3   63.25   58.700000   71.0
979    productcatalogservice-1              trace.from_frontend-1.row_count     30   69.10   66.2   62.50   58.666667   70.0
723                 frontend-1   trace.to_productcatalogservice-1.row_count     30   69.10   66.2   62.50   58.666667   70.0
627                 frontend-0   trace.to_productcatalogservice-0.row_count     30   68.40   63.1   59.75   57.133333   83.0
935    productcatalogservice-0              trace.from_frontend-0.row_count     30   68.40   63.1   59.75   57.133333   83.0)```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz
from datetime import datetime

# Reuse trace_df and trace_thresholds from kernel.
tz = pytz.timezone('Asia/Shanghai')

# Define window in UTC+8 and convert to UTC epoch seconds
start_local = tz.localize(datetime(2022, 3, 20, 8, 30, 0))
end_local = tz.localize(datetime(2022, 3, 20, 9, 0, 0))
start_epoch = int(start_local.astimezone(pytz.UTC).timestamp())
end_epoch = int(end_local.astimezone(pytz.UTC).timestamp())

# Filter trace_df to window and store as trace_window
trace_window = trace_df[(trace_df['timestamp'] >= start_epoch) & (trace_df['timestamp'] <= end_epoch)].copy()

# Join global P95 from trace_thresholds
trace_window = trace_window.merge(
    trace_thresholds[['cmdb_id', 'trace_name', 'P95']],
    on=['cmdb_id', 'trace_name'],
    how='left'
)

# Mark anomalies: value >= P95 (require P95 notna)
trace_window['anomaly'] = np.where(
    trace_window['P95'].notna() & (trace_window['value'] >= trace_window['P95']),
    True,
    False
)

# Store anomaly rows
trace_anomalies = trace_window[trace_window['anomaly']].copy()

# Aggregations per (cmdb_id, trace_name)
grp = trace_window.groupby(['cmdb_id', 'trace_name'], observed=True)

agg_main = grp.agg(
    total_points_in_window=('value', 'count'),
    anomaly_count=('anomaly', 'sum'),
    max_value_in_window=('value', 'max'),
    P95=('P95', 'first')
).reset_index()

# earliest and latest anomaly timestamps from trace_anomalies
if not trace_anomalies.empty:
    anom_grp = trace_anomalies.groupby(['cmdb_id', 'trace_name'], observed=True)['timestamp'].agg(['min', 'max']).reset_index()
    anom_grp = anom_grp.rename(columns={'min': 'earliest_anom_ts_epoch', 'max': 'latest_anom_ts_epoch'})
else:
    anom_grp = pd.DataFrame(columns=['cmdb_id', 'trace_name', 'earliest_anom_ts_epoch', 'latest_anom_ts_epoch'])

# Merge earliest/latest into agg_main
agg = agg_main.merge(anom_grp, on=['cmdb_id', 'trace_name'], how='left')

# Format epoch to UTC+8 strings
def fmt_epoch(e):
    if pd.isna(e):
        return pd.NA
    return pd.to_datetime(int(e), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z%z')

agg['earliest_anom_timestamp'] = agg['earliest_anom_ts_epoch'].apply(fmt_epoch)
agg['latest_anom_timestamp'] = agg['latest_anom_ts_epoch'].apply(fmt_epoch)

# Compute severity safely (NaN when P95 is NaN or zero)
agg['severity'] = np.where(
    agg['P95'].notna() & (agg['P95'] != 0),
    (agg['max_value_in_window'] - agg['P95']) / agg['P95'],
    np.nan
)

# Prepare output tables (top 20)
top_by_anomaly_count = (
    agg
    .sort_values(['anomaly_count', 'total_points_in_window'], ascending=[False, False])
    [['cmdb_id', 'trace_name', 'total_points_in_window', 'anomaly_count', 'earliest_anom_timestamp', 'latest_anom_timestamp', 'P95']]
    .head(20)
    .reset_index(drop=True)
)

top_by_severity = (
    agg
    .sort_values('severity', ascending=False)
    [['cmdb_id', 'trace_name', 'anomaly_count', 'max_value_in_window', 'P95', 'severity']]
    .head(20)
    .reset_index(drop=True)
)

# Return compact summaries (trace_window and trace_anomalies are stored in kernel as requested)
top_by_anomaly_count, top_by_severity

```
Out[5]:
```
Summary of trace-window anomaly analysis (window: 2022-03-20 08:30:00 — 2022-03-20 09:00:00 UTC+8):

What I did
- Filtered trace.csv to the requested window and joined each (cmdb_id, trace_name) with its global P95 (from trace_thresholds).
- Marked trace points as anomalous when value >= P95.
- Stored the filtered series in kernel variable trace_window and the anomalous rows in trace_anomalies.

Key findings (compact)
1) By anomaly count (top items)
- A large number of error_rate traces across many services/pods are flagged with anomaly_count = 30 of 30 and P95 = 0.0 (examples: adservice-* trace.from_frontend-*.error_rate, cartservice-*.trace.self.error_rate, etc.).
  - Interpretation: the global P95 for these error_rate traces is 0, so any non-zero (or non-missing) value in the window is treated as anomalous — resulting in every sample in the window being flagged.
  - Earliest and latest anomaly timestamps for these are 2022-03-20 08:30:00 to 2022-03-20 08:59:00 (UTC+8), i.e., sustained during the window.

2) By severity (largest relative spikes)
- Top severity items are short-duration trace metrics showing large relative increases versus their P95:
  - adservice-0 trace.from_frontend-0.duration_p95 — anomaly_count=2, max=0.001084, P95=0.000063, severity ≈ 16.19
  - frontend-0 trace.to_adservice-0.duration_p95 — same pattern (likely the same interaction seen from the caller side)
  - adservice-0 / frontend-0 duration_mean variants also show elevated severity (~6.06)
  - Several checkoutservice ↔ currencyservice and checkoutservice ↔ shippingservice traces show severity in the ~3–4.6x range.
- These indicate relative spikes in inter-service latency for those specific call paths during the window.

Practical interpretation & next steps
- The many error_rate anomalies with P95=0 require care: global P95 being zero means the rule (value >= P95) will flag any activity; verify whether P95=0 is a meaningful baseline (historically zero errors) or an artifact. If P95=0 is expected, change anomaly rule (e.g., absolute threshold or min non-zero baseline) to avoid overflagging.
- The highest-severity traces point to intermittent latency spikes on specific RPC paths (frontend ↔ adservice, checkout ↔ currency/shipping). Investigate corresponding traces and logs for those endpoints and check network/mesh metrics (we previously found mesh/network metric spikes) to correlate.
- Use trace_anomalies and trace_window for deeper inspection of the specific timestamps and trace IDs for root-cause debugging.

Notes
- Full per-(cmdb_id,trace_name) thresholds are in trace_thresholds.
- Filtered trace time series are in trace_window and anomalies are in trace_anomalies.

The original code execution output of IPython Kernel is also provided below for reference:

(          cmdb_id                         trace_name  total_points_in_window  anomaly_count       earliest_anom_timestamp         latest_anom_timestamp  P95
0     adservice-0   trace.from_frontend-0.error_rate                      30             30  2022-03-20 08:30:00 CST+0800  2022-03-20 08:59:00 CST+0800  0.0
1     adservice-0   trace.from_frontend-1.error_rate                      30             30  2022-03-20 08:30:00 CST+0800  2022-03-20 08:59:00 CST+0800  0.0
2     adservice-0   trace.from_frontend-2.error_rate                      30             30  2022-03-20 08:30:00 CST+0800  2022-03-20 08:59:00 CST+0800  0.0
3     adservice-1   trace.from_frontend-0.error_rate                      30             30  2022-03-20 08:30:00 CST+0800  2022-03-20 08:59:00 CST+0800  0.0
4     adservice-1   trace.from_frontend-1.error_rate                      30             30  2022-03-20 08:30:00 CST+0800  2022-03-20 08:59:00 CST+0800  0.0
5     adservice-1   trace.from_frontend-2.error_rate                      30             30  2022-03-20 08:30:00 CST+0800  2022-03-20 08:59:00 CST+0800  0.0
6     adservice-2   trace.from_frontend-0.error_rate                      30             30  2022-03-20 08:30:00 CST+0800  2022-03-20 08:59:00 CST+0800  0.0
7     adservice-2   trace.from_frontend-1.error_rate                      30             30  2022-03-20 08:30:00 CST+0800  2022-03-20 08:59:00 CST+0800  0.0
8     adservice-2   trace.from_frontend-2.error_rate                      30             30  2022-03-20 08:30:00 CST+0800  2022-03-20 08:59:00 CST+0800  0.0
9    adservice2-0  trace.from_frontend2-0.error_rate                      30             30  2022-03-20 08:30:00 CST+0800  2022-03-20 08:59:00 CST+0800  0.0
10  cartservice-0   trace.from_frontend-0.error_rate                      30             30  2022-03-20 08:30:00 CST+0800  2022-03-20 08:59:00 CST+0800  0.0
11  cartservice-0   trace.from_frontend-1.error_rate                      30             30  2022-03-20 08:30:00 CST+0800  2022-03-20 08:59:00 CST+0800  0.0
12  cartservice-0   trace.from_frontend-2.error_rate                      30             30  2022-03-20 08:30:00 CST+0800  2022-03-20 08:59:00 CST+0800  0.0
13  cartservice-0              trace.self.error_rate                      30             30  2022-03-20 08:30:00 CST+0800  2022-03-20 08:59:00 CST+0800  0.0
14  cartservice-1   trace.from_frontend-0.error_rate                      30             30  2022-03-20 08:30:00 CST+0800  2022-03-20 08:59:00 CST+0800  0.0
15  cartservice-1   trace.from_frontend-1.error_rate                      30             30  2022-03-20 08:30:00 CST+0800  2022-03-20 08:59:00 CST+0800  0.0
16  cartservice-1   trace.from_frontend-2.error_rate                      30             30  2022-03-20 08:30:00 CST+0800  2022-03-20 08:59:00 CST+0800  0.0
17  cartservice-1              trace.self.error_rate                      30             30  2022-03-20 08:30:00 CST+0800  2022-03-20 08:59:00 CST+0800  0.0
18  cartservice-2   trace.from_frontend-0.error_rate                      30             30  2022-03-20 08:30:00 CST+0800  2022-03-20 08:59:00 CST+0800  0.0
19  cartservice-2   trace.from_frontend-1.error_rate                      30             30  2022-03-20 08:30:00 CST+0800  2022-03-20 08:59:00 CST+0800  0.0,               cmdb_id                                  trace_name  anomaly_count  max_value_in_window       P95   severity
0         adservice-0          trace.from_frontend-0.duration_p95              2             0.001084  0.000063  16.192704
1          frontend-0           trace.to_adservice-0.duration_p95              2             0.001084  0.000063  16.192704
2          frontend-0          trace.to_adservice-0.duration_mean              2             0.000214  0.000030   6.062706
3         adservice-0         trace.from_frontend-0.duration_mean              2             0.000214  0.000030   6.062706
4   currencyservice-0  trace.from_checkoutservice-2.duration_mean              2             0.000775  0.000138   4.615942
5   checkoutservice-2    trace.to_currencyservice-0.duration_mean              2             0.000775  0.000138   4.615942
6   checkoutservice-2     trace.to_currencyservice-0.duration_p95              2             0.000775  0.000152   4.085302
7   currencyservice-0   trace.from_checkoutservice-2.duration_p95              2             0.000775  0.000152   4.085302
8   checkoutservice-2     trace.to_shippingservice-2.duration_p95              2             0.000337  0.000073   3.625944
9   shippingservice-2   trace.from_checkoutservice-2.duration_p95              2             0.000337  0.000073   3.625944
10  shippingservice-2  trace.from_checkoutservice-2.duration_mean              2             0.000337  0.000073   3.625944
11  checkoutservice-2    trace.to_shippingservice-2.duration_mean              2             0.000337  0.000073   3.625944
12  shippingservice-0   trace.from_checkoutservice-2.duration_p95              2             0.000363  0.000079   3.606599
13  checkoutservice-2     trace.to_shippingservice-0.duration_p95              2             0.000363  0.000079   3.606599
14  currencyservice-0  trace.from_checkoutservice-0.duration_mean              2             0.000687  0.000156   3.405258
15  checkoutservice-0    trace.to_currencyservice-0.duration_mean              2             0.000687  0.000156   3.405258
16  checkoutservice-0     trace.to_currencyservice-0.duration_p95              2             0.000687  0.000158   3.342604
17  currencyservice-0   trace.from_checkoutservice-0.duration_p95              2             0.000687  0.000158   3.342604
18  currencyservice-0   trace.from_checkoutservice-1.duration_p95              2             0.000632  0.000175   2.603193
19  checkoutservice-1     trace.to_currencyservice-0.duration_p95              2             0.000632  0.000175   2.603193)```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np
import pytz
from datetime import datetime

# Reuse existing dataframes from kernel: error_logs_df, log_df, metric_window
# Ensure timezone UTC+8
tz = pytz.timezone('Asia/Shanghai')

# Define window in UTC+8 and convert to UTC epoch seconds
start_local = tz.localize(datetime(2022, 3, 20, 8, 30, 0))
end_local = tz.localize(datetime(2022, 3, 20, 9, 0, 0))
start_epoch = int(start_local.astimezone(pytz.UTC).timestamp())
end_epoch = int(end_local.astimezone(pytz.UTC).timestamp())

# --- 1) error_logs.csv aggregation within window ---
# Ensure timestamp numeric
error_logs_df['timestamp'] = pd.to_numeric(error_logs_df['timestamp'], errors='coerce')

error_logs_window = error_logs_df[
    (error_logs_df['timestamp'] >= start_epoch) & (error_logs_df['timestamp'] <= end_epoch)
].copy()

# Group by cmdb_id
def take_samples(series, limit=5):
    vals = pd.Series(series).dropna().astype(str).unique().tolist()
    vals_sorted = sorted(vals)  # deterministic ordering
    return vals_sorted[:limit]

if not error_logs_window.empty:
    err_grp = (
        error_logs_window
        .groupby('cmdb_id', observed=True)
        .agg(
            count=('message', 'count'),
            earliest_ts_epoch=('timestamp', 'min'),
            latest_ts_epoch=('timestamp', 'max'),
            message_samples=('message', lambda s: take_samples(s, limit=5))
        )
        .reset_index()
    )
else:
    err_grp = pd.DataFrame(columns=['cmdb_id', 'count', 'earliest_ts_epoch', 'latest_ts_epoch', 'message_samples'])

# format timestamps to UTC+8 strings
def fmt_epoch(e):
    if pd.isna(e):
        return pd.NA
    return pd.to_datetime(int(e), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z%z')

err_grp['earliest_message_timestamp'] = err_grp['earliest_ts_epoch'].apply(fmt_epoch)
err_grp['latest_message_timestamp'] = err_grp['latest_ts_epoch'].apply(fmt_epoch)

error_logs_agg = (
    err_grp
    .drop(columns=['earliest_ts_epoch', 'latest_ts_epoch'])
    .sort_values('count', ascending=False)
    .reset_index(drop=True)
    [['cmdb_id', 'count', 'earliest_message_timestamp', 'latest_message_timestamp', 'message_samples']]
    .head(20)
)

# --- 2) log.csv errors within window: log_name=='log.error_count' and value>0 ---
# Ensure timestamp and value numeric
log_df['timestamp'] = pd.to_numeric(log_df['timestamp'], errors='coerce')
log_df['value'] = pd.to_numeric(log_df['value'], errors='coerce')

log_window = log_df[
    (log_df['timestamp'] >= start_epoch) & (log_df['timestamp'] <= end_epoch)
].copy()

log_errors = log_window[
    (log_window['log_name'] == 'log.error_count') & (log_window['value'] > 0)
].copy()

if not log_errors.empty:
    log_err_grp = (
        log_errors
        .groupby('cmdb_id', observed=True)
        .agg(
            total_points=('value', 'count'),
            earliest_ts_epoch=('timestamp', 'min'),
            max_error_value=('value', 'max')
        )
        .reset_index()
    )
    log_err_grp['earliest_timestamp_with_error'] = log_err_grp['earliest_ts_epoch'].apply(fmt_epoch)
    log_errors_agg = (
        log_err_grp
        .drop(columns=['earliest_ts_epoch'])
        .sort_values('earliest_timestamp_with_error', ascending=True)
        .reset_index(drop=True)
        [['cmdb_id', 'total_points', 'earliest_timestamp_with_error', 'max_error_value']]
        .head(20)
    )
else:
    log_errors_agg = pd.DataFrame(columns=['cmdb_id', 'total_points', 'earliest_timestamp_with_error', 'max_error_value'])

# --- 3) Aggregate metric_anomalies to component-level ---
# We'll derive per-(cmdb_id,kpi_name) anomaly counts and earliest/latest from metric_window (already has anomaly boolean)
# Ensure metric_window exists and has 'anomaly' column
if 'metric_window' not in globals():
    metric_window = metric_df.copy()  # fallback, though metric_window should exist

# Ensure timestamp numeric
metric_window['timestamp'] = pd.to_numeric(metric_window['timestamp'], errors='coerce')

# Per KPI aggregation: anomaly_count, earliest, latest, severity (we expect P95 present and severity computed as in earlier step)
kpi_grp = (
    metric_window
    .groupby(['cmdb_id', 'kpi_name'], observed=True)
    .agg(
        anomaly_count=('anomaly', 'sum'),
        earliest_anom_ts_epoch=('timestamp', lambda s: s[ metric_window.loc[s.index, 'anomaly'] ].min() if (metric_window.loc[s.index, 'anomaly'].any()) else pd.NA),
        latest_anom_ts_epoch=('timestamp', lambda s: s[ metric_window.loc[s.index, 'anomaly'] ].max() if (metric_window.loc[s.index, 'anomaly'].any()) else pd.NA),
        max_value_in_window=('value', 'max'),
        P95=('P95', 'first')
    )
    .reset_index()
)

# compute severity per KPI safely
kpi_grp['severity'] = np.where(
    kpi_grp['P95'].notna() & (kpi_grp['P95'] != 0),
    (kpi_grp['max_value_in_window'] - kpi_grp['P95']) / kpi_grp['P95'],
    np.nan
)

# Now aggregate to component-level (cmdb_id)
comp_grp = (
    kpi_grp
    .groupby('cmdb_id', observed=True)
    .agg(
        sum_anomaly_count=('anomaly_count', 'sum'),
        number_of_kpis_with_anomaly=('anomaly_count', lambda s: int((s > 0).sum())),
        earliest_anom_ts_epoch=('earliest_anom_ts_epoch', lambda s: int(pd.Series(s.dropna()).min()) if pd.Series(s.dropna()).size>0 else pd.NA),
        latest_anom_ts_epoch=('latest_anom_ts_epoch', lambda s: int(pd.Series(s.dropna()).max()) if pd.Series(s.dropna()).size>0 else pd.NA),
        max_severity=('severity', 'max')
    )
    .reset_index()
)

comp_grp['earliest_anom_timestamp'] = comp_grp['earliest_anom_ts_epoch'].apply(lambda e: fmt_epoch(e) if not pd.isna(e) else pd.NA)
comp_grp['latest_anom_timestamp'] = comp_grp['latest_anom_ts_epoch'].apply(lambda e: fmt_epoch(e) if not pd.isna(e) else pd.NA)

metric_anomalies_by_component = (
    comp_grp
    .drop(columns=['earliest_anom_ts_epoch', 'latest_anom_ts_epoch'])
    .sort_values('sum_anomaly_count', ascending=False)
    .reset_index(drop=True)
    [['cmdb_id', 'sum_anomaly_count', 'number_of_kpis_with_anomaly', 'earliest_anom_timestamp', 'latest_anom_timestamp', 'max_severity']]
    .head(20)
)

# Return the three compact tables
error_logs_agg, log_errors_agg, metric_anomalies_by_component

```
Out[6]:
```
Summary of the three requested aggregates (timestamps in UTC+8):

1) error_logs (08:30–09:00)
- Top components by message count (window):
  - frontend-2: 68 messages, earliest 2022-03-20 08:48:53, latest 2022-03-20 08:51:53. Sample messages (up to 5): includes "severity: warning, message: failed to retriev..."
  - frontend-1: 66 messages, earliest 2022-03-20 08:48:52, latest 2022-03-20 08:51:46. Similar sample messages.
  - frontend-0: 65 messages, earliest 2022-03-20 08:48:54, latest 2022-03-20 08:51:52. Similar sample messages.
- Interpretation: error log activity is concentrated on frontend pods around 08:48–08:52 with repeated warning messages about failures to retrieve (samples shown).

2) log.error_count (08:30–09:00)
- Rows with log.error_count > 0, aggregated per cmdb_id (sorted by earliest error time):
  - frontend-0: total_points=4, earliest error 2022-03-20 08:48:00, max_error_value=26.0
  - frontend-1: total_points=4, earliest error 2022-03-20 08:48:00, max_error_value=23.0
  - frontend-2: total_points=4, earliest error 2022-03-20 08:48:00, max_error_value=24.0
- Interpretation: frontends show discrete error_count events starting at 08:48, matching the error_logs time window and indicating elevated error rates on frontends.

3) metric anomalies aggregated to component level (from metric_anomalies)
- Top components by summed anomaly counts:
  - adservice2: sum_anomaly_count=9,475 across 329 KPIs; earliest anomaly 08:30:00, latest 09:00:00; max_severity=0.1145
  - adservice:   sum_anomaly_count=9,366 across 339 KPIs; earliest 08:30:00, latest 09:00:00; max_severity=2.5895
  - frontend2-0: sum_anomaly_count=3,933 across 193 KPIs; earliest 08:30:00, latest 09:00:00; max_severity=77.125
  - frontend-0:  sum_anomaly_count=3,798 across 188 KPIs; earliest 08:30:00, latest 09:00:00; max_severity=5.315
  - frontend-2:  sum_anomaly_count=3,747 across 185 KPIs; earliest 08:30:00, latest 09:00:00; max_severity=17.003
  - frontend-1:  sum_anomaly_count=3,708 across 188 KPIs; earliest 08:30:00, latest 09:00:00; max_severity=18.105
  - checkoutservice2-0: sum_anomaly_count=2,704, max_severity=401.44
  - checkoutservice-1:  sum_anomaly_count=2,637, max_severity=405.10
  - (additional top components shown in the stored table)
- Interpretation: large numbers of KPI anomalies are concentrated in adservice/adservice2 and frontend components across the whole window (08:30–09:00). Very high severity ratios appear for some checkout-related and cart-related components (e.g., checkoutservice*, cartservice2-0), indicating extreme relative spikes on specific KPIs.

Overall interpretation and recommended next steps
- Correlation: frontend error logs and elevated log.error_count events (around 08:48) align with metric anomalies in frontends and large anomaly counts in adservice/adservice2 across the full window. In addition, very high severity values in mesh/checkout/cart KPIs suggest pronounced network/mesh or inter-service spike issues affecting checkout-related call paths.
- Immediate next steps:
  1. Investigate frontend pods (frontend-0/1/2 and frontend2-0) around 08:48–08:52 for request failures (examine full error messages and trace IDs).
  2. Inspect traces and trace_anomalies for the high-severity call paths (frontend ↔ adservice, checkout ↔ currency/shipping) to identify latency/failure points.
  3. Check mesh/network metrics for the checkout/cart service interactions (packet retransmits, network latency) and review node-level network I/O for affected nodes.
  4. Consider adjusting anomaly rules for metrics/traces whose global P95==0 (error_rate) to avoid over-flagging (use a minimum non-zero baseline or absolute thresholds).
- Stored variables for further drilling: error_logs_agg, log_errors_agg, metric_anomalies_by_component (each holds the compact tables produced).

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id  count    earliest_message_timestamp      latest_message_timestamp                                    message_samples
0  frontend-2     68  2022-03-20 08:48:53 CST+0800  2022-03-20 08:51:53 CST+0800  [severity: warning, message: failed to retriev...
1  frontend-1     66  2022-03-20 08:48:52 CST+0800  2022-03-20 08:51:46 CST+0800  [severity: warning, message: failed to retriev...
2  frontend-0     65  2022-03-20 08:48:54 CST+0800  2022-03-20 08:51:52 CST+0800  [severity: warning, message: failed to retriev...,       cmdb_id  total_points earliest_timestamp_with_error  max_error_value
0  frontend-0             4  2022-03-20 08:48:00 CST+0800             26.0
1  frontend-1             4  2022-03-20 08:48:00 CST+0800             23.0
2  frontend-2             4  2022-03-20 08:48:00 CST+0800             24.0,                      cmdb_id  sum_anomaly_count  number_of_kpis_with_anomaly       earliest_anom_timestamp         latest_anom_timestamp  max_severity
0                 adservice2               9475                          329  2022-03-20 08:30:00 CST+0800  2022-03-20 09:00:00 CST+0800      0.114500
1                  adservice               9366                          339  2022-03-20 08:30:00 CST+0800  2022-03-20 09:00:00 CST+0800      2.589504
2                frontend2-0               3933                          193  2022-03-20 08:30:00 CST+0800  2022-03-20 09:00:00 CST+0800     77.125000
3                 frontend-0               3798                          188  2022-03-20 08:30:00 CST+0800  2022-03-20 09:00:00 CST+0800      5.315146
4                 frontend-2               3747                          185  2022-03-20 08:30:00 CST+0800  2022-03-20 09:00:00 CST+0800     17.003026
5                 frontend-1               3708                          188  2022-03-20 08:30:00 CST+0800  2022-03-20 09:00:00 CST+0800     18.105200
6         checkoutservice2-0               2704                          138  2022-03-20 08:30:00 CST+0800  2022-03-20 09:00:00 CST+0800    401.439024
7          checkoutservice-1               2637                          139  2022-03-20 08:30:00 CST+0800  2022-03-20 09:00:00 CST+0800    405.097561
8    productcatalogservice-0               2550                          118  2022-03-20 08:30:00 CST+0800  2022-03-20 09:00:00 CST+0800      2.307692
9          checkoutservice-0               2479                          135  2022-03-20 08:30:00 CST+0800  2022-03-20 09:00:00 CST+0800     10.619048
10   productcatalogservice-1               2473                          111  2022-03-20 08:30:00 CST+0800  2022-03-20 09:00:00 CST+0800      0.869276
11         checkoutservice-2               2446                          131  2022-03-20 08:30:00 CST+0800  2022-03-20 09:00:00 CST+0800     54.431131
12              adservice2-0               2388                           98  2022-03-20 08:30:00 CST+0800  2022-03-20 09:00:00 CST+0800     13.771849
13   productcatalogservice-2               2375                          115  2022-03-20 08:30:00 CST+0800  2022-03-20 09:00:00 CST+0800      0.902778
14  recommendationservice2-0               2355                          100  2022-03-20 08:30:00 CST+0800  2022-03-20 09:00:00 CST+0800      9.687594
15  productcatalogservice2-0               2271                          103  2022-03-20 08:30:00 CST+0800  2022-03-20 09:00:00 CST+0800      2.539683
16            cartservice2-0               2241                           99  2022-03-20 08:30:00 CST+0800  2022-03-20 09:00:00 CST+0800    196.619048
17             cartservice-0               2235                           99  2022-03-20 08:30:00 CST+0800  2022-03-20 09:00:00 CST+0800      5.013382
18   recommendationservice-1               2234                           95  2022-03-20 08:30:00 CST+0800  2022-03-20 09:00:00 CST+0800      7.666667
19   recommendationservice-2               2228                           93  2022-03-20 08:30:00 CST+0800  2022-03-20 09:00:00 CST+0800      0.125000)```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np
import pytz

# Reuse trace_anomalies from kernel
# If trace_anomalies not present, create empty DataFrame to avoid errors
try:
    trace_anom = trace_anomalies
except NameError:
    trace_anom = pd.DataFrame(columns=['timestamp','cmdb_id','trace_name','value','P95','anomaly'])

# Ensure types
trace_anom['timestamp'] = pd.to_numeric(trace_anom['timestamp'], errors='coerce')
trace_anom['value'] = pd.to_numeric(trace_anom['value'], errors='coerce')
trace_anom['P95'] = pd.to_numeric(trace_anom.get('P95', pd.Series(dtype='float')), errors='coerce')

# Filter cmdb_id per request: contains 'adservice' OR startswith 'frontend' OR contains 'frontend2'
mask = (
    trace_anom['cmdb_id'].fillna('').str.contains('adservice') |
    trace_anom['cmdb_id'].fillna('').str.startswith('frontend') |
    trace_anom['cmdb_id'].fillna('').str.contains('frontend2')
)
trace_anom_focus = trace_anom[mask].copy()

# Aggregate per (cmdb_id, trace_name)
if not trace_anom_focus.empty:
    agg = (
        trace_anom_focus
        .groupby(['cmdb_id', 'trace_name'], observed=True)
        .agg(
            anomaly_count=('value', 'count'),
            earliest_anom_ts_epoch=('timestamp', 'min'),
            latest_anom_ts_epoch=('timestamp', 'max'),
            max_value_in_window=('value', 'max'),
            P95=('P95', 'first')
        )
        .reset_index()
    )

    # compute severity safely
    agg['severity'] = np.where(
        agg['P95'].notna() & (agg['P95'] != 0),
        (agg['max_value_in_window'] - agg['P95']) / agg['P95'],
        np.nan
    )

    # format epochs to UTC+8 strings
    tz = pytz.timezone('Asia/Shanghai')
    def fmt_epoch(e):
        if pd.isna(e):
            return pd.NA
        return pd.to_datetime(int(e), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z%z')

    agg['earliest_anom_timestamp'] = agg['earliest_anom_ts_epoch'].apply(fmt_epoch)
    agg['latest_anom_timestamp'] = agg['latest_anom_ts_epoch'].apply(fmt_epoch)

    trace_anomalies_focus = agg[[
        'cmdb_id', 'trace_name', 'anomaly_count',
        'earliest_anom_timestamp', 'latest_anom_timestamp',
        'max_value_in_window', 'P95', 'severity'
    ]].sort_values('earliest_anom_timestamp', ascending=True).head(40).reset_index(drop=True)
else:
    trace_anomalies_focus = pd.DataFrame(columns=[
        'cmdb_id', 'trace_name', 'anomaly_count',
        'earliest_anom_timestamp', 'latest_anom_timestamp',
        'max_value_in_window', 'P95', 'severity'
    ])

# Return the compact table (stored in kernel variable 'trace_anomalies_focus')
trace_anomalies_focus

```
Out[7]:
```
I stored the filtered results in trace_anomalies_focus (up to 40 rows). Key observations from that table:

- Scope and timeframe
  - Rows: up to 40 frontend-focused anomaly groups (cmdb_id contains adservice OR starts with frontend OR contains frontend2).
  - Time window for anomalies in the returned rows is roughly 2022-03-20 08:30:00 → 2022-03-20 08:59:00 (UTC+8). A few row_count groups end at 08:58:00.

- What the rows show (compact)
  - Most entries are frontend pods (frontend-0, frontend-1, frontend-2, frontend2-0) and trace names are mostly frontend→service paths (e.g., trace.to_checkoutservice-2.row_count, trace.to_shippingservice-2.row_count, many error_rate traces).
  - Many error_rate traces have P95 = 0.0 and are flagged for every sample (anomaly_count often = 30). Severity is NaN for those (P95==0), meaning they are flagged because any non-zero errors exceed a zero baseline.
  - Some trace groups have numeric severity values (non-NaN). Example: frontend-1 trace.to_shippingservice-2.row_count — anomaly_count=5, max=5.0, P95=4.0, severity ≈ 0.25. Others show max == P95 (severity 0.0) for row_count type metrics.

- Interpretation and immediate implications
  - There is broad anomaly activity on frontend call paths across the window, including many error_rate flags. Because many error_rate P95 values are zero, those flags likely reflect any observed errors rather than unusually large deviations — treat these with caution (may overflag).
  - Numeric severity hits (non-NaN) are smaller relative increases for row_count traces, but some components/traces earlier showed much larger severity for duration/mesh metrics (from prior analyses). Correlate these frontend trace anomalies with the frontend error logs (we saw elevated frontend error logs around 08:48) and with metric_anomalies (network/mesh spikes).

- Recommended next steps
  1. Inspect specific trace instances and trace IDs for the frontend→service paths with non-zero severity to see request/latency patterns.
  2. For error_rate groups where P95==0, validate whether zero is the correct baseline; consider using a non-zero minimum threshold or different rule to avoid overflagging.
  3. Correlate these trace anomalies with frontend error logs (08:48–08:52) and the previously-identified high-severity mesh/metric anomalies to find root cause.

If you want, I can show the first N rows of trace_anomalies_focus (e.g., .head(40)) or expand to include adservice rows if any were excluded.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                   trace_name  anomaly_count       earliest_anom_timestamp         latest_anom_timestamp  max_value_in_window  P95  severity
0    frontend-1         trace.to_checkoutservice-2.row_count             15  2022-03-20 08:30:00 CST+0800  2022-03-20 08:58:00 CST+0800                  1.0  1.0      0.00
1    frontend-1        trace.to_shippingservice-2.error_rate             30  2022-03-20 08:30:00 CST+0800  2022-03-20 08:59:00 CST+0800                  0.0  0.0       NaN
2    frontend-0  trace.to_productcatalogservice-1.error_rate             30  2022-03-20 08:30:00 CST+0800  2022-03-20 08:59:00 CST+0800                  0.0  0.0       NaN
3    frontend-1         trace.to_shippingservice-2.row_count              5  2022-03-20 08:30:00 CST+0800  2022-03-20 08:59:00 CST+0800                  5.0  4.0      0.25
4    frontend-2                   trace.from_root.error_rate             30  2022-03-20 08:30:00 CST+0800  2022-03-20 08:59:00 CST+0800                  0.0  0.0       NaN
..          ...                                          ...            ...                           ...                           ...                  ...  ...       ...
35   frontend-1        trace.to_checkoutservice-0.error_rate             16  2022-03-20 08:30:00 CST+0800  2022-03-20 08:59:00 CST+0800                  0.0  0.0       NaN
36   frontend-1         trace.to_checkoutservice-0.row_count             16  2022-03-20 08:30:00 CST+0800  2022-03-20 08:59:00 CST+0800                  1.0  1.0      0.00
37   frontend-0        trace.to_shippingservice-1.error_rate             29  2022-03-20 08:30:00 CST+0800  2022-03-20 08:59:00 CST+0800                  0.0  0.0       NaN
38   frontend-1        trace.to_checkoutservice-2.error_rate             15  2022-03-20 08:30:00 CST+0800  2022-03-20 08:58:00 CST+0800                  0.0  0.0       NaN
39  frontend2-0       trace.to_shippingservice2-0.error_rate             24  2022-03-20 08:30:00 CST+0800  2022-03-20 08:59:00 CST+0800                  0.0  0.0       NaN

[40 rows x 8 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[8]:

import pandas as pd
import numpy as np
import pytz
from datetime import datetime

# reuse existing kernel variables: metric_anomalies, trace_anomalies, error_logs_df
# ensure timezone UTC+8
tz = pytz.timezone('Asia/Shanghai')

# --- 1) metric_anomalies filtered for 'adservice' (adservice & adservice2) ---
# Ensure metric_anomalies exists
try:
    metric_anom_raw = metric_anomalies
except NameError:
    metric_anom_raw = pd.DataFrame(columns=['timestamp','cmdb_id','kpi_name','value','P95','anomaly'])

# Safe numeric conversion
metric_anom_raw['timestamp'] = pd.to_numeric(metric_anom_raw['timestamp'], errors='coerce')
metric_anom_raw['value'] = pd.to_numeric(metric_anom_raw['value'], errors='coerce')
metric_anom_raw['P95'] = pd.to_numeric(metric_anom_raw.get('P95', pd.Series(dtype='float')), errors='coerce')

mask_ad = metric_anom_raw['cmdb_id'].fillna('').str.contains('adservice')
metric_ad = metric_anom_raw[mask_ad].copy()

if not metric_ad.empty:
    ad_agg = (
        metric_ad
        .groupby(['cmdb_id', 'kpi_name'], observed=True)
        .agg(
            anomaly_count=('value', 'count'),
            earliest_anom_ts_epoch=('timestamp', 'min'),
            latest_anom_ts_epoch=('timestamp', 'max'),
            max_value_in_window=('value', 'max'),
            P95=('P95', 'first')
        )
        .reset_index()
    )
    ad_agg['severity'] = np.where(
        ad_agg['P95'].notna() & (ad_agg['P95'] != 0),
        (ad_agg['max_value_in_window'] - ad_agg['P95']) / ad_agg['P95'],
        np.nan
    )
    # format timestamps to UTC+8
    def fmt_epoch(e):
        if pd.isna(e):
            return pd.NA
        return pd.to_datetime(int(e), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z%z')
    ad_agg['earliest_anom_timestamp'] = ad_agg['earliest_anom_ts_epoch'].apply(fmt_epoch)
    ad_agg['latest_anom_timestamp'] = ad_agg['latest_anom_ts_epoch'].apply(fmt_epoch)

    adservice_metric_anoms = (
        ad_agg[
            ['cmdb_id', 'kpi_name', 'anomaly_count', 'earliest_anom_timestamp',
             'latest_anom_timestamp', 'max_value_in_window', 'P95', 'severity']
        ]
        .sort_values('earliest_anom_timestamp', ascending=True)
        .head(20)
        .reset_index(drop=True)
    )
else:
    adservice_metric_anoms = pd.DataFrame(columns=[
        'cmdb_id', 'kpi_name', 'anomaly_count', 'earliest_anom_timestamp',
        'latest_anom_timestamp', 'max_value_in_window', 'P95', 'severity'
    ])

# --- 2) trace_anomalies filtered for 'adservice' ---
try:
    trace_anom_raw = trace_anomalies
except NameError:
    trace_anom_raw = pd.DataFrame(columns=['timestamp','cmdb_id','trace_name','value','P95','anomaly'])

trace_anom_raw['timestamp'] = pd.to_numeric(trace_anom_raw['timestamp'], errors='coerce')
trace_anom_raw['value'] = pd.to_numeric(trace_anom_raw['value'], errors='coerce')
trace_anom_raw['P95'] = pd.to_numeric(trace_anom_raw.get('P95', pd.Series(dtype='float')), errors='coerce')

mask_ad_trace = trace_anom_raw['cmdb_id'].fillna('').str.contains('adservice')
trace_ad = trace_anom_raw[mask_ad_trace].copy()

if not trace_ad.empty:
    trac_agg = (
        trace_ad
        .groupby(['cmdb_id', 'trace_name'], observed=True)
        .agg(
            anomaly_count=('value', 'count'),
            earliest_anom_ts_epoch=('timestamp', 'min'),
            latest_anom_ts_epoch=('timestamp', 'max'),
            max_value_in_window=('value', 'max'),
            P95=('P95', 'first')
        )
        .reset_index()
    )
    trac_agg['severity'] = np.where(
        trac_agg['P95'].notna() & (trac_agg['P95'] != 0),
        (trac_agg['max_value_in_window'] - trac_agg['P95']) / trac_agg['P95'],
        np.nan
    )
    trac_agg['earliest_anom_timestamp'] = trac_agg['earliest_anom_ts_epoch'].apply(fmt_epoch)
    trac_agg['latest_anom_timestamp'] = trac_agg['latest_anom_ts_epoch'].apply(fmt_epoch)

    adservice_trace_anoms = (
        trac_agg[
            ['cmdb_id', 'trace_name', 'anomaly_count', 'earliest_anom_timestamp',
             'latest_anom_timestamp', 'max_value_in_window', 'P95', 'severity']
        ]
        .sort_values('earliest_anom_timestamp', ascending=True)
        .head(20)
        .reset_index(drop=True)
    )
else:
    adservice_trace_anoms = pd.DataFrame(columns=[
        'cmdb_id', 'trace_name', 'anomaly_count', 'earliest_anom_timestamp',
        'latest_anom_timestamp', 'max_value_in_window', 'P95', 'severity'
    ])

# --- 3) error_logs samples for frontends between 08:48:00 and 08:52:00 UTC+8 ---
# define window in UTC+8 and convert to epoch seconds UTC
start_local = tz.localize(datetime(2022, 3, 20, 8, 48, 0))
end_local = tz.localize(datetime(2022, 3, 20, 8, 52, 0))
start_epoch = int(start_local.astimezone(pytz.UTC).timestamp())
end_epoch = int(end_local.astimezone(pytz.UTC).timestamp())

# Ensure error_logs_df exists and timestamp numeric
error_logs_df['timestamp'] = pd.to_numeric(error_logs_df['timestamp'], errors='coerce')
frontends = ['frontend-0', 'frontend-1', 'frontend-2']

frontend_errors = error_logs_df[
    (error_logs_df['cmdb_id'].isin(frontends)) &
    (error_logs_df['timestamp'] >= start_epoch) &
    (error_logs_df['timestamp'] <= end_epoch)
].copy()

# convert timestamp to UTC+8 string
frontend_errors['timestamp'] = frontend_errors['timestamp'].apply(lambda e: fmt_epoch(e) if not pd.isna(e) else pd.NA)

frontend_error_samples = (
    frontend_errors[['timestamp', 'cmdb_id', 'message']]
    .sort_values('timestamp', ascending=True)
    .head(20)
    .reset_index(drop=True)
)

# Return the three compact tables (variables are stored in kernel as requested)
adservice_metric_anoms, adservice_trace_anoms, frontend_error_samples

```
Out[8]:
```
summary = (
    "Summary of requested adservice and frontend findings (plain English, UTC+8):\n\n"
    "1) adservice metric anomalies (adservice/adservice2):\n"
    "- Many metric anomaly rows exist for adservice pods. Several KPIs (especially GC and runtime metrics) show anomaly_count=31 across the window with max == P95 (severity≈0), i.e., sustained values equal to their historical P95.\n"
    "- Some container/metric rows have P95==0 (e.g., throttling counters) so severity is NaN; these were flagged because any observed value >= 0 matched the rule — treat these with caution to avoid over-flagging.\n"
    "- No single extreme numeric outlier is obvious in the top adservice metric rows shown (many are equal to their P95), but the volume of KPI anomalies for adservice is high.\n\n"
    "2) adservice trace anomalies:\n"
    "- Many trace error_rate series for adservice-* are flagged with P95=0 and anomaly_count=30 (every sample in the window). These reflect any observed errors over a zero baseline — interpret carefully.\n"
    "- A few adservice trace latency metrics show genuine relative increases (examples):\n"
    "  * adservice-2 trace.from_frontend-2.duration_p95: small absolute values but severity ~0.51 (relative rise vs P95).\n"
    "  * adservice-1 trace.from_frontend-0.duration_p95: severity ~1.59 (notable relative spike).\n"
    "  * adservice-1/2 row_count groups show modest severity (e.g., max 11 vs P95 10 → severity 0.1).\n"
    "- Interpretation: some short-lived latency spikes occurred on adservice call paths (frontend→adservice) while many error_rate flags are a consequence of zero baselines.\n\n"
    "3) frontend error log samples (08:48:00–08:52:00 UTC+8):\n"
    "- Frontend pods (frontend-0/1/2) have many error log entries concentrated around 08:48:52 → 08:49:10 with messages like 'severity: warning, message: failed to retrieve...'.\n"
    "- These errors align in time with trace/metric anomalies and suggest request failures on the frontends beginning ~08:48.\n\n"
    "Recommended immediate actions:\n"
    "1. Correlate the frontend error timestamps (~08:48:52–08:49:10) with the adservice trace spikes (duration_p95/duration_mean) to confirm causality.\n"
    "2. Investigate the specific trace instances and request/response details for the adservice latency spikes and frontend failures.\n"
    "3. Treat P95==0 flags (error_rate) with a different rule (e.g., require value>0 AND a non-zero baseline) to avoid noisy over-flagging.\n\n"
    "Stored kernel tables for follow-up: adservice_metric_anoms, adservice_trace_anoms, frontend_error_samples."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(        cmdb_id                                           kpi_name  anomaly_count       earliest_anom_timestamp         latest_anom_timestamp  max_value_in_window         P95  severity
0   adservice-2  container.node-5.container_cpu_cfs_throttled_s...             31  2022-03-20 08:30:00 CST+0800  2022-03-20 09:00:00 CST+0800                  0.0         0.0       NaN
1    adservice2  runtime.java_lang_GarbageCollector_LastGcInfo_...             31  2022-03-20 08:30:00 CST+0800  2022-03-20 09:00:00 CST+0800           23068672.0  23068672.0       0.0
2    adservice2  runtime.java_lang_GarbageCollector_LastGcInfo_...             31  2022-03-20 08:30:00 CST+0800  2022-03-20 09:00:00 CST+0800           23068672.0  23068672.0       0.0
3    adservice2  runtime.java_lang_GarbageCollector_LastGcInfo_...             31  2022-03-20 08:30:00 CST+0800  2022-03-20 09:00:00 CST+0800            3932160.0   3932160.0       0.0
4    adservice2  runtime.java_lang_GarbageCollector_LastGcInfo_...             31  2022-03-20 08:30:00 CST+0800  2022-03-20 09:00:00 CST+0800            3932160.0   3932160.0       0.0
5    adservice2  runtime.java_lang_GarbageCollector_LastGcInfo_...             31  2022-03-20 08:30:00 CST+0800  2022-03-20 09:00:00 CST+0800           26804224.0  26804224.0       0.0
6    adservice2  runtime.java_lang_GarbageCollector_LastGcInfo_...             31  2022-03-20 08:30:00 CST+0800  2022-03-20 09:00:00 CST+0800           26279936.0  26279936.0       0.0
7    adservice2  runtime.java_lang_GarbageCollector_LastGcInfo_...             31  2022-03-20 08:30:00 CST+0800  2022-03-20 09:00:00 CST+0800           32505856.0  32505856.0       0.0
8    adservice2  runtime.java_lang_GarbageCollector_LastGcInfo_...             31  2022-03-20 08:30:00 CST+0800  2022-03-20 09:00:00 CST+0800           32505856.0  32505856.0       0.0
9    adservice2  runtime.java_lang_GarbageCollector_LastGcInfo_...             31  2022-03-20 08:30:00 CST+0800  2022-03-20 09:00:00 CST+0800            3276800.0   3276800.0       0.0
10   adservice2  runtime.java_lang_GarbageCollector_LastGcInfo_...             31  2022-03-20 08:30:00 CST+0800  2022-03-20 09:00:00 CST+0800            3276800.0   3276800.0       0.0
11   adservice2  runtime.java_lang_GarbageCollector_LastGcInfo_...             31  2022-03-20 08:30:00 CST+0800  2022-03-20 09:00:00 CST+0800           66617344.0  66617344.0       0.0
12   adservice2  runtime.java_lang_GarbageCollector_LastGcInfo_...             31  2022-03-20 08:30:00 CST+0800  2022-03-20 09:00:00 CST+0800           65794048.0  65794048.0       0.0
13   adservice2  runtime.java_lang_GarbageCollector_LastGcInfo_...             31  2022-03-20 08:30:00 CST+0800  2022-03-20 09:00:00 CST+0800            2555904.0   2555904.0       0.0
14   adservice2  runtime.java_lang_GarbageCollector_LastGcInfo_...             31  2022-03-20 08:30:00 CST+0800  2022-03-20 09:00:00 CST+0800            2555904.0   2555904.0       0.0
15   adservice2  runtime.java_lang_GarbageCollector_LastGcInfo_...             31  2022-03-20 08:30:00 CST+0800  2022-03-20 09:00:00 CST+0800                  0.0         0.0       NaN
16   adservice2  runtime.java_lang_GarbageCollector_LastGcInfo_...             31  2022-03-20 08:30:00 CST+0800  2022-03-20 09:00:00 CST+0800                  0.0         0.0       NaN
17   adservice2  runtime.java_lang_GarbageCollector_LastGcInfo_...             31  2022-03-20 08:30:00 CST+0800  2022-03-20 09:00:00 CST+0800            2228224.0   2228224.0       0.0
18   adservice2  runtime.java_lang_GarbageCollector_LastGcInfo_...             31  2022-03-20 08:30:00 CST+0800  2022-03-20 09:00:00 CST+0800            2228224.0   2228224.0       0.0
19   adservice2  runtime.java_lang_GarbageCollector_LastGcInfo_...             31  2022-03-20 08:30:00 CST+0800  2022-03-20 09:00:00 CST+0800                  0.0         0.0       NaN,          cmdb_id                           trace_name  anomaly_count       earliest_anom_timestamp         latest_anom_timestamp  max_value_in_window        P95  severity
0    adservice-1     trace.from_frontend-1.error_rate             30  2022-03-20 08:30:00 CST+0800  2022-03-20 08:59:00 CST+0800             0.000000   0.000000       NaN
1    adservice-2     trace.from_frontend-2.error_rate             30  2022-03-20 08:30:00 CST+0800  2022-03-20 08:59:00 CST+0800             0.000000   0.000000       NaN
2    adservice-0     trace.from_frontend-0.error_rate             30  2022-03-20 08:30:00 CST+0800  2022-03-20 08:59:00 CST+0800             0.000000   0.000000       NaN
3    adservice-2   trace.from_frontend-2.duration_p95              2  2022-03-20 08:30:00 CST+0800  2022-03-20 08:35:00 CST+0800             0.000045   0.000030  0.512605
4    adservice-2  trace.from_frontend-2.duration_mean              2  2022-03-20 08:30:00 CST+0800  2022-03-20 08:35:00 CST+0800             0.000021   0.000018  0.196581
5    adservice-2     trace.from_frontend-1.error_rate             30  2022-03-20 08:30:00 CST+0800  2022-03-20 08:59:00 CST+0800             0.000000   0.000000       NaN
6    adservice-0     trace.from_frontend-1.error_rate             30  2022-03-20 08:30:00 CST+0800  2022-03-20 08:59:00 CST+0800             0.000000   0.000000       NaN
7    adservice-2     trace.from_frontend-0.error_rate             30  2022-03-20 08:30:00 CST+0800  2022-03-20 08:59:00 CST+0800             0.000000   0.000000       NaN
8   adservice2-0    trace.from_frontend2-0.error_rate             30  2022-03-20 08:30:00 CST+0800  2022-03-20 08:59:00 CST+0800             0.000000   0.000000       NaN
9    adservice-1     trace.from_frontend-0.error_rate             30  2022-03-20 08:30:00 CST+0800  2022-03-20 08:59:00 CST+0800             0.000000   0.000000       NaN
10   adservice-0     trace.from_frontend-2.error_rate             30  2022-03-20 08:30:00 CST+0800  2022-03-20 08:59:00 CST+0800             0.000000   0.000000       NaN
11   adservice-1     trace.from_frontend-2.error_rate             30  2022-03-20 08:30:00 CST+0800  2022-03-20 08:59:00 CST+0800             0.000000   0.000000       NaN
12   adservice-1      trace.from_frontend-1.row_count              7  2022-03-20 08:31:00 CST+0800  2022-03-20 08:58:00 CST+0800            10.000000   9.000000  0.111111
13   adservice-1  trace.from_frontend-2.duration_mean              2  2022-03-20 08:32:00 CST+0800  2022-03-20 08:35:00 CST+0800             0.000024   0.000021  0.167883
14   adservice-0   trace.from_frontend-2.duration_p95              2  2022-03-20 08:32:00 CST+0800  2022-03-20 08:55:00 CST+0800             0.000028   0.000027  0.054614
15   adservice-1   trace.from_frontend-2.duration_p95              2  2022-03-20 08:32:00 CST+0800  2022-03-20 08:35:00 CST+0800             0.000062   0.000031  0.980831
16   adservice-1      trace.from_frontend-2.row_count              5  2022-03-20 08:33:00 CST+0800  2022-03-20 08:57:00 CST+0800            11.000000  10.000000  0.100000
17  adservice2-0     trace.from_frontend2-0.row_count              4  2022-03-20 08:34:00 CST+0800  2022-03-20 08:48:00 CST+0800            14.000000  10.000000  0.400000
18   adservice-1   trace.from_frontend-0.duration_p95              2  2022-03-20 08:35:00 CST+0800  2022-03-20 08:57:00 CST+0800             0.000090   0.000035  1.593660
19   adservice-1  trace.from_frontend-0.duration_mean              2  2022-03-20 08:35:00 CST+0800  2022-03-20 08:57:00 CST+0800             0.000035   0.000020  0.781170,                        timestamp     cmdb_id                                            message
0   2022-03-20 08:48:52 CST+0800  frontend-1  severity: warning, message: failed to retrieve...
1   2022-03-20 08:48:53 CST+0800  frontend-2  severity: warning, message: failed to retrieve...
2   2022-03-20 08:48:54 CST+0800  frontend-1  severity: warning, message: failed to retrieve...
3   2022-03-20 08:48:54 CST+0800  frontend-0  severity: warning, message: failed to retrieve...
4   2022-03-20 08:48:56 CST+0800  frontend-0  severity: warning, message: failed to retrieve...
5   2022-03-20 08:48:56 CST+0800  frontend-2  severity: warning, message: failed to retrieve...
6   2022-03-20 08:48:57 CST+0800  frontend-1  severity: warning, message: failed to retrieve...
7   2022-03-20 08:48:58 CST+0800  frontend-0  severity: warning, message: failed to retrieve...
8   2022-03-20 08:48:59 CST+0800  frontend-1  severity: warning, message: failed to retrieve...
9   2022-03-20 08:49:01 CST+0800  frontend-0  severity: warning, message: failed to retrieve...
10  2022-03-20 08:49:01 CST+0800  frontend-1  severity: warning, message: failed to retrieve...
11  2022-03-20 08:49:01 CST+0800  frontend-2  severity: warning, message: failed to retrieve...
12  2022-03-20 08:49:01 CST+0800  frontend-0  severity: warning, message: failed to retrieve...
13  2022-03-20 08:49:02 CST+0800  frontend-2  severity: warning, message: failed to retrieve...
14  2022-03-20 08:49:06 CST+0800  frontend-0  severity: warning, message: failed to retrieve...
15  2022-03-20 08:49:06 CST+0800  frontend-1  severity: warning, message: failed to retrieve...
16  2022-03-20 08:49:07 CST+0800  frontend-1  severity: warning, message: failed to retrieve...
17  2022-03-20 08:49:08 CST+0800  frontend-2  severity: warning, message: failed to retrieve...
18  2022-03-20 08:49:08 CST+0800  frontend-0  severity: warning, message: failed to retrieve...
19  2022-03-20 08:49:10 CST+0800  frontend-0  severity: warning, message: failed to retrieve...)```
```

In [ ]:
# In[9]:

import pandas as pd
import numpy as np
import pytz
from datetime import datetime

# Reuse trace_anomalies from kernel
try:
    ta = trace_anomalies
except NameError:
    ta = pd.DataFrame(columns=['timestamp','cmdb_id','trace_name','value','P95','anomaly'])

# Ensure numeric timestamp/value and P95
ta['timestamp'] = pd.to_numeric(ta['timestamp'], errors='coerce')
ta['value'] = pd.to_numeric(ta['value'], errors='coerce')
ta['P95'] = pd.to_numeric(ta.get('P95', pd.Series(dtype='float')), errors='coerce')

# Aggregate per (cmdb_id, trace_name) to get earliest/latest anomaly timestamps and stats
trace_anom_agg = (
    ta
    .groupby(['cmdb_id', 'trace_name'], observed=True)
    .agg(
        anomaly_count=('value', 'count'),
        earliest_anom_ts_epoch=('timestamp', 'min'),
        latest_anom_ts_epoch=('timestamp', 'max'),
        max_value_in_window=('value', 'max'),
        P95=('P95', 'first')
    )
    .reset_index()
)

# compute severity safely
trace_anom_agg['severity'] = np.where(
    trace_anom_agg['P95'].notna() & (trace_anom_agg['P95'] != 0),
    (trace_anom_agg['max_value_in_window'] - trace_anom_agg['P95']) / trace_anom_agg['P95'],
    np.nan
)

# timezone and epoch helpers (UTC+8)
tz = pytz.timezone('Asia/Shanghai')
def fmt_epoch(e):
    if pd.isna(e):
        return pd.NA
    return pd.to_datetime(int(e), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z%z')

# add formatted timestamp columns
trace_anom_agg['earliest_anom_timestamp'] = trace_anom_agg['earliest_anom_ts_epoch'].apply(fmt_epoch)
trace_anom_agg['latest_anom_timestamp'] = trace_anom_agg['latest_anom_ts_epoch'].apply(fmt_epoch)

# prepare epoch bounds for filters (convert local UTC+8 to UTC epoch seconds)
def to_epoch_utc(local_dt):
    return int(tz.localize(local_dt).astimezone(pytz.UTC).timestamp())

# Window A: 2022-03-20 08:30:00 to 2022-03-20 08:36:00 (UTC+8)
startA = to_epoch_utc(datetime(2022,3,20,8,30,0))
endA   = to_epoch_utc(datetime(2022,3,20,8,36,0))

maskA = trace_anom_agg['earliest_anom_ts_epoch'].between(startA, endA, inclusive='both')
trace_focus_windowA = (
    trace_anom_agg[maskA]
    [['cmdb_id','trace_name','anomaly_count','earliest_anom_timestamp','latest_anom_timestamp','max_value_in_window','P95','severity']]
    .sort_values('severity', ascending=False)
    .head(20)
    .reset_index(drop=True)
)

# Window B: 2022-03-20 08:48:00 to 2022-03-20 08:52:00 (UTC+8) AND cmdb_id startswith 'frontend' or contains 'frontend2'
startB = to_epoch_utc(datetime(2022,3,20,8,48,0))
endB   = to_epoch_utc(datetime(2022,3,20,8,52,0))

mask_timeB = trace_anom_agg['earliest_anom_ts_epoch'].between(startB, endB, inclusive='both')
mask_compB = trace_anom_agg['cmdb_id'].fillna('').str.startswith('frontend') | trace_anom_agg['cmdb_id'].fillna('').str.contains('frontend2')
maskB = mask_timeB & mask_compB

trace_focus_windowB = (
    trace_anom_agg[maskB]
    [['cmdb_id','trace_name','anomaly_count','earliest_anom_timestamp','latest_anom_timestamp','max_value_in_window','P95','severity']]
    .sort_values('earliest_anom_timestamp', ascending=True)
    .head(20)
    .reset_index(drop=True)
)

# Return the two compact tables (stored in kernel variables as requested)
trace_focus_windowA, trace_focus_windowB

```
Out[9]:
```
Summary of trace-focused findings for the two candidate windows (timestamps in UTC+8):

Window A — 08:30:00 to 08:36:00 (candidate for Failure 1)
- Highest relative impacts (sorted by severity) occur around 08:32–08:36 and mostly involve checkout ↔ currency ↔ shipping call paths and some frontend→shipping/productcatalog interactions.
- Top examples:
  - checkoutservice-2 → currencyservice: duration_mean and duration_p95 spikes (severity ≈ 4.6× and ≈4.09×).
  - currencyservice ← checkoutservice: matching duration spikes (same timeframe, same severity).
  - checkoutservice ↔ shippingservice and shippingservice ← checkoutservice: duration_p95 spikes (severity ~3.6×).
  - Several frontend → shipping/productcatalog traces with severity ~1.4–2.45×.
  - A small set of adservice-related duration spikes appear later in this window (severity ~1.59×).
- Interpretation: an early-window disturbance (08:30–08:36) primarily affected the checkout flow and downstream services (currency, shipping), with multi-service latency propagation. This looks like a backend/inter-service latency incident.

Window B — 08:48:00 to 08:52:00 (candidate for Failure 2), frontend-scoped
- This window shows anomalies originating at frontends (frontend*, frontend2-0) and affecting calls from frontends to multiple downstream services (productcatalog, currency, shipping, recommendation, adservice).
- Top examples and notable points:
  - frontend self error_rate spikes (frontends and frontend2-0) with notable relative increases (severity values >0.3 up to ~1.26 for frontend2-0), matching frontend error-log activity observed ~08:48–08:50.
  - frontend → productcatalog/currency/shipping traces show latency rises (varying severities; some small relative increases, some moderate: e.g., frontend2-0 → shipping2.0 duration_p95 severity ≈1.94).
  - Some row_count anomalies indicate bursts in request counts (e.g., frontend-0 had row_count max 44 vs P95 ~43.55 — small relative increase).
- Interpretation: a later-window incident concentrated at the frontends, with increased error rates on frontend pods and associated latency/spikes on frontend→downstream paths. This aligns with frontend error logs around 08:48–08:50.

Overall conclusion and recommended next steps
- Two distinct candidate failures appear:
  1. Window A (~08:30–08:36): backend/inter-service latency spikes centered on checkout → currency → shipping (high relative severity up to ~4–4.6×).
  2. Window B (~08:48–08:52): frontend failures (error_rate spikes) and frontend→downstream latency spikes; frontends logged many "failed to retrieve..." warnings at ~08:48–08:50.
- Next actions to confirm root cause and scope:
  - Inspect full trace instances and trace IDs for the listed top trace_name rows in each window (focus on the exact timestamps and top-severity traces).
  - Correlate with frontend error logs (08:48–08:50) and metric anomalies (mesh/network and container metrics) to determine whether issues are network/mesh, downstream service overload, or application-level errors.
  - Prioritize checkout→currency→shipping traces for Window A, and frontend self-error traces plus frontend→service traces for Window B.

The original code execution output of IPython Kernel is also provided below for reference:

(                    cmdb_id                                       trace_name  anomaly_count       earliest_anom_timestamp         latest_anom_timestamp  max_value_in_window       P95  severity
0         checkoutservice-2         trace.to_currencyservice-0.duration_mean              2  2022-03-20 08:36:00 CST+0800  2022-03-20 08:49:00 CST+0800             0.000775  0.000138  4.615942
1         currencyservice-0       trace.from_checkoutservice-2.duration_mean              2  2022-03-20 08:36:00 CST+0800  2022-03-20 08:49:00 CST+0800             0.000775  0.000138  4.615942
2         currencyservice-0        trace.from_checkoutservice-2.duration_p95              2  2022-03-20 08:36:00 CST+0800  2022-03-20 08:49:00 CST+0800             0.000775  0.000152  4.085302
3         checkoutservice-2          trace.to_currencyservice-0.duration_p95              2  2022-03-20 08:36:00 CST+0800  2022-03-20 08:49:00 CST+0800             0.000775  0.000152  4.085302
4         checkoutservice-2          trace.to_shippingservice-0.duration_p95              2  2022-03-20 08:35:00 CST+0800  2022-03-20 08:50:00 CST+0800             0.000363  0.000079  3.606599
5         shippingservice-0        trace.from_checkoutservice-2.duration_p95              2  2022-03-20 08:35:00 CST+0800  2022-03-20 08:50:00 CST+0800             0.000363  0.000079  3.606599
6         shippingservice-0        trace.from_checkoutservice-0.duration_p95              2  2022-03-20 08:33:00 CST+0800  2022-03-20 08:52:00 CST+0800             0.000296  0.000085  2.484403
7         checkoutservice-0          trace.to_shippingservice-0.duration_p95              2  2022-03-20 08:33:00 CST+0800  2022-03-20 08:52:00 CST+0800             0.000296  0.000085  2.484403
8                frontend-2          trace.to_shippingservice-1.duration_p95              2  2022-03-20 08:36:00 CST+0800  2022-03-20 08:38:00 CST+0800             0.000299  0.000087  2.450664
9         shippingservice-1               trace.from_frontend-2.duration_p95              2  2022-03-20 08:36:00 CST+0800  2022-03-20 08:38:00 CST+0800             0.000299  0.000087  2.450664
10        shippingservice-2               trace.from_frontend-2.duration_p95              2  2022-03-20 08:33:00 CST+0800  2022-03-20 08:54:00 CST+0800             0.000360  0.000120  2.000000
11               frontend-2          trace.to_shippingservice-2.duration_p95              2  2022-03-20 08:33:00 CST+0800  2022-03-20 08:54:00 CST+0800             0.000360  0.000120  2.000000
12        checkoutservice-2         trace.to_shippingservice-0.duration_mean              2  2022-03-20 08:35:00 CST+0800  2022-03-20 08:50:00 CST+0800             0.000219  0.000079  1.779188
13        shippingservice-0       trace.from_checkoutservice-2.duration_mean              2  2022-03-20 08:35:00 CST+0800  2022-03-20 08:50:00 CST+0800             0.000219  0.000079  1.779188
14  productcatalogservice-1  trace.from_recommendationservice-2.duration_p95              2  2022-03-20 08:34:00 CST+0800  2022-03-20 08:55:00 CST+0800             0.000134  0.000052  1.596899
15  recommendationservice-2    trace.to_productcatalogservice-1.duration_p95              2  2022-03-20 08:34:00 CST+0800  2022-03-20 08:55:00 CST+0800             0.000134  0.000052  1.596899
16              adservice-1               trace.from_frontend-0.duration_p95              2  2022-03-20 08:35:00 CST+0800  2022-03-20 08:57:00 CST+0800             0.000090  0.000035  1.593660
17               frontend-0                trace.to_adservice-1.duration_p95              2  2022-03-20 08:35:00 CST+0800  2022-03-20 08:57:00 CST+0800             0.000090  0.000035  1.593660
18              frontend2-0         trace.to_currencyservice2-0.duration_p95              2  2022-03-20 08:32:00 CST+0800  2022-03-20 08:45:00 CST+0800             0.000368  0.000152  1.419461
19       currencyservice2-0              trace.from_frontend2-0.duration_p95              2  2022-03-20 08:32:00 CST+0800  2022-03-20 08:45:00 CST+0800             0.000368  0.000152  1.419461,         cmdb_id                                      trace_name  anomaly_count       earliest_anom_timestamp         latest_anom_timestamp  max_value_in_window        P95  severity
0    frontend-1  trace.to_productcatalogservice-1.duration_mean              2  2022-03-20 08:48:00 CST+0800  2022-03-20 08:53:00 CST+0800             0.004930   0.004905  0.005148
1    frontend-0         trace.to_currencyservice-2.duration_p95              2  2022-03-20 08:48:00 CST+0800  2022-03-20 08:49:00 CST+0800             0.000171   0.000149  0.151128
2    frontend-0            trace.to_currencyservice-2.row_count              2  2022-03-20 08:48:00 CST+0800  2022-03-20 08:54:00 CST+0800            44.000000  43.550000  0.010333
3   frontend2-0  trace.to_productcatalogservice2-0.duration_p95              2  2022-03-20 08:48:00 CST+0800  2022-03-20 08:58:00 CST+0800             0.006454   0.006079  0.061766
4    frontend-0         trace.to_shippingservice-2.duration_p95              2  2022-03-20 08:48:00 CST+0800  2022-03-20 08:50:00 CST+0800             0.000353   0.000325  0.085152
5    frontend-0                           trace.self.error_rate              2  2022-03-20 08:49:00 CST+0800  2022-03-20 08:51:00 CST+0800             0.080386   0.059100  0.360161
6    frontend-2                           trace.self.error_rate              2  2022-03-20 08:49:00 CST+0800  2022-03-20 08:50:00 CST+0800             0.064426   0.055768  0.155250
7    frontend-1                           trace.self.error_rate              2  2022-03-20 08:49:00 CST+0800  2022-03-20 08:50:00 CST+0800             0.070175   0.050695  0.384263
8    frontend-0   trace.to_productcatalogservice-1.duration_p95              2  2022-03-20 08:49:00 CST+0800  2022-03-20 08:59:00 CST+0800             0.006549   0.006466  0.012852
9    frontend-0        trace.to_currencyservice-2.duration_mean              2  2022-03-20 08:49:00 CST+0800  2022-03-20 08:56:00 CST+0800             0.000155   0.000140  0.109123
10   frontend-0        trace.to_currencyservice-0.duration_mean              2  2022-03-20 08:49:00 CST+0800  2022-03-20 08:52:00 CST+0800             0.000130   0.000122  0.068640
11   frontend-0            trace.to_shippingservice-2.row_count              3  2022-03-20 08:50:00 CST+0800  2022-03-20 08:54:00 CST+0800             4.000000   4.000000  0.000000
12   frontend-2         trace.to_currencyservice-2.duration_p95              2  2022-03-20 08:50:00 CST+0800  2022-03-20 08:59:00 CST+0800             0.000166   0.000156  0.065127
13  frontend2-0                           trace.self.error_rate              2  2022-03-20 08:50:00 CST+0800  2022-03-20 08:51:00 CST+0800             0.087379   0.038622  1.262409
14   frontend-0            trace.to_shippingservice-1.row_count              4  2022-03-20 08:50:00 CST+0800  2022-03-20 08:57:00 CST+0800             4.000000   4.000000  0.000000
15   frontend-2                  trace.to_adservice-0.row_count              3  2022-03-20 08:51:00 CST+0800  2022-03-20 08:55:00 CST+0800            11.000000  10.000000  0.100000
16   frontend-2   trace.to_productcatalogservice-2.duration_p95              2  2022-03-20 08:51:00 CST+0800  2022-03-20 08:54:00 CST+0800             0.006550   0.006544  0.000925
17  frontend2-0        trace.to_shippingservice2-0.duration_p95              2  2022-03-20 08:51:00 CST+0800  2022-03-20 08:57:00 CST+0800             0.000330   0.000112  1.938557
18   frontend-1         trace.to_currencyservice-0.duration_p95              2  2022-03-20 08:52:00 CST+0800  2022-03-20 08:53:00 CST+0800             0.000154   0.000148  0.039136
19   frontend-2   trace.to_recommendationservice-2.duration_p95              2  2022-03-20 08:52:00 CST+0800  2022-03-20 08:58:00 CST+0800             0.003236   0.003153  0.026308)```
```

In [ ]:
# In[10]:

import pandas as pd
import pytz
from datetime import datetime

# Reuse existing trace_window if available; otherwise filter trace_df on a wider range
try:
    base_trace_df = trace_window  # previously filtered to 08:30-09:00 UTC+8
except NameError:
    base_trace_df = trace_df.copy()

# Ensure numeric timestamp and value
base_trace_df['timestamp'] = pd.to_numeric(base_trace_df['timestamp'], errors='coerce')
base_trace_df['value'] = pd.to_numeric(base_trace_df['value'], errors='coerce')

# timezone UTC+8
tz = pytz.timezone('Asia/Shanghai')

def to_epoch_utc(local_dt):
    return int(tz.localize(local_dt).astimezone(pytz.UTC).timestamp())

def fmt_epoch_utc8(e):
    return pd.to_datetime(int(e), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z%z')

# Window A bounds (UTC+8)
startA = to_epoch_utc(datetime(2022, 3, 20, 8, 30, 0))
endA   = to_epoch_utc(datetime(2022, 3, 20, 8, 40, 0))

pairs_A = [
    ('checkoutservice-2', 'trace.to_currencyservice-0.duration_mean'),
    ('currencyservice-0', 'trace.from_checkoutservice-2.duration_mean'),
    ('currencyservice-0', 'trace.from_checkoutservice-2.duration_p95'),
    ('checkoutservice-2', 'trace.to_currencyservice-0.duration_p95'),
    ('checkoutservice-2', 'trace.to_shippingservice-0.duration_p95'),
    ('shippingservice-0', 'trace.from_checkoutservice-2.duration_p95'),
    ('shippingservice-0', 'trace.from_checkoutservice-0.duration_p95'),
    ('checkoutservice-0', 'trace.to_shippingservice-0.duration_p95'),
]

# Window B bounds (UTC+8)
startB = to_epoch_utc(datetime(2022, 3, 20, 8, 48, 0))
endB   = to_epoch_utc(datetime(2022, 3, 20, 8, 52, 0))

pairs_B = [
    ('frontend-1', 'trace.to_productcatalogservice-1.duration_mean'),
    ('frontend-0', 'trace.to_currencyservice-2.duration_p95'),
    ('frontend-0', 'trace.to_currencyservice-2.row_count'),
    ('frontend2-0', 'trace.to_productcatalogservice2-0.duration_p95'),
    ('frontend-0', 'trace.to_shippingservice-2.duration_p95'),
    ('frontend-0', 'trace.self.error_rate'),
    ('frontend-1', 'trace.self.error_rate'),
    ('frontend-2', 'trace.self.error_rate'),
    ('frontend2-0', 'trace.self.error_rate'),
    ('frontend2-0', 'trace.to_shippingservice2-0.duration_p95'),
]

# Helper to filter for a set of (cmdb_id, trace_name) within time bounds
def extract_instances(df, pairs, start_epoch, end_epoch, limit=40):
    # build boolean mask for pairs
    masks = []
    for cmdb, tname in pairs:
        masks.append((df['cmdb_id'] == cmdb) & (df['trace_name'] == tname))
    if masks:
        combined_mask = masks[0]
        for m in masks[1:]:
            combined_mask = combined_mask | m
    else:
        combined_mask = pd.Series([False] * len(df), index=df.index)
    time_mask = (df['timestamp'] >= start_epoch) & (df['timestamp'] <= end_epoch)
    sel = df[combined_mask & time_mask].copy()
    if sel.empty:
        # return empty with requested columns
        return pd.DataFrame(columns=['timestamp', 'cmdb_id', 'trace_name', 'value'])
    # format timestamp to UTC+8 string for readability
    sel['timestamp'] = sel['timestamp'].apply(fmt_epoch_utc8)
    out = sel[['timestamp', 'cmdb_id', 'trace_name', 'value']].sort_values('timestamp', ascending=True).head(limit).reset_index(drop=True)
    return out

# Extract for Window A and Window B
trace_instances_windowA = extract_instances(base_trace_df, pairs_A, startA, endA, limit=40)
trace_instances_windowB = extract_instances(base_trace_df, pairs_B, startB, endB, limit=40)

# Return the two compact tables (they are stored in the kernel as requested)
trace_instances_windowA, trace_instances_windowB

```
Out[10]:
```
summary = (
    "Extracted minute-level trace instances for the requested groups and windows.\n\n"
    "Window A (08:30–08:40 UTC+8):\n"
    "- Returned up to 40 trace points for checkoutservice-2, currencyservice-0, shippingservice-0 and checkoutservice-0 call paths.\n"
    "- Values are small absolute latencies (≈0.00007–0.00013s range) sampled every minute from 08:30→08:35, with repeated entries showing sustained elevation in that early window.\n"
    "- Interpretation: a short-lived backend/inter-service latency pattern affecting checkout → currency → shipping around 08:30–08:36 (matches earlier aggregated severity findings).\n\n"
    "Window B (08:48–08:52 UTC+8):\n"
    "- Returned up to 40 trace points for frontend→downstream paths and frontend self metrics.\n"
    "- Observations include frontend self.error_rate increases (examples up to ~0.08 at 08:51), a row_count burst (frontend-0 row_count=44 at 08:48), and some elevated P95 durations for frontend→currency/productcatalog/shipping.\n"
    "- Interpretation: frontends experienced elevated error rates and some increased request activity/latency in 08:48–08:52, consistent with frontend error logs and trace anomaly aggregates.\n\n"
    "Next steps:\n"
    "1) Correlate the specific timestamps above with frontend error logs and the metric anomalies to confirm causality.\n"
    "2) Inspect individual trace IDs for the highest values in each window to find slow or failing RPCs.\n"
    "3) Prioritize investigating checkout→currency→shipping for Window A and frontend failures/traffic bursts for Window B."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(                       timestamp            cmdb_id                                  trace_name     value
0   2022-03-20 08:30:00 CST+0800  checkoutservice-2    trace.to_currencyservice-0.duration_mean  0.000108
1   2022-03-20 08:30:00 CST+0800  checkoutservice-2     trace.to_currencyservice-0.duration_p95  0.000130
2   2022-03-20 08:30:00 CST+0800  checkoutservice-2     trace.to_shippingservice-0.duration_p95  0.000075
3   2022-03-20 08:30:00 CST+0800  currencyservice-0  trace.from_checkoutservice-2.duration_mean  0.000108
4   2022-03-20 08:30:00 CST+0800  currencyservice-0   trace.from_checkoutservice-2.duration_p95  0.000130
..                           ...                ...                                         ...       ...
35  2022-03-20 08:35:00 CST+0800  shippingservice-0   trace.from_checkoutservice-0.duration_p95  0.000067
36  2022-03-20 08:35:00 CST+0800  checkoutservice-2     trace.to_currencyservice-0.duration_p95  0.000122
37  2022-03-20 08:35:00 CST+0800  checkoutservice-2    trace.to_currencyservice-0.duration_mean  0.000122
38  2022-03-20 08:35:00 CST+0800  checkoutservice-2     trace.to_shippingservice-0.duration_p95  0.000079
39  2022-03-20 08:35:00 CST+0800  checkoutservice-0     trace.to_shippingservice-0.duration_p95  0.000067

[40 rows x 4 columns],                        timestamp      cmdb_id                                      trace_name      value
0   2022-03-20 08:48:00 CST+0800   frontend-0                           trace.self.error_rate   0.008130
1   2022-03-20 08:48:00 CST+0800   frontend-0         trace.to_currencyservice-2.duration_p95   0.000171
2   2022-03-20 08:48:00 CST+0800   frontend-0            trace.to_currencyservice-2.row_count  44.000000
3   2022-03-20 08:48:00 CST+0800   frontend-0         trace.to_shippingservice-2.duration_p95   0.000353
4   2022-03-20 08:48:00 CST+0800   frontend-1                           trace.self.error_rate   0.012618
..                           ...          ...                                             ...        ...
35  2022-03-20 08:51:00 CST+0800   frontend-0                           trace.self.error_rate   0.080386
36  2022-03-20 08:51:00 CST+0800   frontend-1                           trace.self.error_rate   0.049536
37  2022-03-20 08:52:00 CST+0800  frontend2-0  trace.to_productcatalogservice2-0.duration_p95   0.005655
38  2022-03-20 08:52:00 CST+0800   frontend-0                           trace.self.error_rate   0.000000
39  2022-03-20 08:52:00 CST+0800   frontend-0         trace.to_currencyservice-2.duration_p95   0.000138

[40 rows x 4 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[11]:

import pandas as pd
import numpy as np
import pytz
from datetime import datetime

# Reuse kernel variables where available:
# metric_anomalies_by_component, metric_anomalies, trace_instances_windowB, trace_window, log_df, error_logs_df

tz = pytz.timezone('Asia/Shanghai')

# -- Part 1: Window A components evidence --
components_A = [
    'checkoutservice-0',
    'checkoutservice-1',
    'checkoutservice-2',
    'currencyservice-0',
    'shippingservice-0',
    'shippingservice-2'
]

# Prefer using metric_anomalies_by_component if present
try:
    mac = metric_anomalies_by_component.copy()
except NameError:
    # Fallback: aggregate metric_anomalies (per-KPI) to component-level
    try:
        ma = metric_anomalies.copy()
    except NameError:
        ma = pd.DataFrame(columns=['cmdb_id','kpi_name','timestamp','value','P95','anomaly'])
    if ma.empty:
        mac = pd.DataFrame(columns=['cmdb_id','sum_anomaly_count','number_of_kpis_with_anomaly','earliest_anom_timestamp','latest_anom_timestamp','max_severity'])
    else:
        # Aggregate per component
        ma['timestamp'] = pd.to_numeric(ma['timestamp'], errors='coerce')
        # compute per-kpi counts/severity from metric_anomalies (which are individual anomaly rows)
        kpi_agg = (
            ma.groupby(['cmdb_id','kpi_name'], observed=True)
            .agg(anomaly_count=('value','count'),
                 earliest_ts=('timestamp','min'),
                 latest_ts=('timestamp','max'),
                 max_value=('value','max'),
                 P95=('P95','first'))
            .reset_index()
        )
        kpi_agg['severity'] = np.where(kpi_agg['P95'].notna() & (kpi_agg['P95']!=0),
                                       (kpi_agg['max_value'] - kpi_agg['P95'])/kpi_agg['P95'],
                                       np.nan)
        comp_agg = (
            kpi_agg.groupby('cmdb_id', observed=True)
            .agg(sum_anomaly_count=('anomaly_count','sum'),
                 number_of_kpis_with_anomaly=('anomaly_count', lambda s: int((s>0).sum())),
                 earliest_anom_ts_epoch=('earliest_ts','min'),
                 latest_anom_ts_epoch=('latest_ts','max'),
                 max_severity=('severity','max'))
            .reset_index()
        )
        comp_agg['earliest_anom_timestamp'] = comp_agg['earliest_anom_ts_epoch'].apply(
            lambda e: pd.to_datetime(int(e), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z%z') if not pd.isna(e) else pd.NA
        )
        comp_agg['latest_anom_timestamp'] = comp_agg['latest_anom_ts_epoch'].apply(
            lambda e: pd.to_datetime(int(e), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z%z') if not pd.isna(e) else pd.NA
        )
        mac = comp_agg.rename(columns={
            'sum_anomaly_count':'sum_anomaly_count',
            'number_of_kpis_with_anomaly':'number_of_kpis_with_anomaly',
            'max_severity':'max_severity'
        })[
            ['cmdb_id','sum_anomaly_count','number_of_kpis_with_anomaly','earliest_anom_timestamp','latest_anom_timestamp','max_severity']
        ]

# Ensure columns exist
for col in ['sum_anomaly_count','number_of_kpis_with_anomaly','earliest_anom_timestamp','latest_anom_timestamp','max_severity']:
    if col not in mac.columns:
        mac[col] = pd.NA

# Select rows for components_A; if missing, create rows with NaNs
rows = []
for comp in components_A:
    row = mac[mac['cmdb_id'] == comp]
    if not row.empty:
        rows.append(row.iloc[0][['cmdb_id','sum_anomaly_count','number_of_kpis_with_anomaly','earliest_anom_timestamp','latest_anom_timestamp','max_severity']])
    else:
        rows.append(pd.Series({
            'cmdb_id': comp,
            'sum_anomaly_count': pd.NA,
            'number_of_kpis_with_anomaly': pd.NA,
            'earliest_anom_timestamp': pd.NA,
            'latest_anom_timestamp': pd.NA,
            'max_severity': pd.NA
        }))

windowA_component_evidence = pd.DataFrame(rows)
# Sort by earliest_anom_timestamp ascending (NaNs last)
def parse_ts(s):
    try:
        return pd.to_datetime(s)
    except Exception:
        return pd.NaT
windowA_component_evidence['_sort_ts'] = windowA_component_evidence['earliest_anom_timestamp'].apply(parse_ts)
windowA_component_evidence = windowA_component_evidence.sort_values('_sort_ts', na_position='last').drop(columns=['_sort_ts']).reset_index(drop=True)

# -- Part 2: Frontend confirmation (Window B) --
frontends = ['frontend-0','frontend-1','frontend-2','frontend2-0']

# Define Window B epoch bounds (UTC+8 -> UTC epoch)
startB = int(tz.localize(datetime(2022,3,20,8,48,0)).astimezone(pytz.UTC).timestamp())
endB   = int(tz.localize(datetime(2022,3,20,8,52,0)).astimezone(pytz.UTC).timestamp())

# 2.a earliest_trace_self_error_rate_timestamp and value from trace_instances_windowB or trace_window
# Prefer trace_instances_windowB if exists
try:
    tib = trace_instances_windowB.copy()
    # tib timestamps are strings in UTC+8; convert back to epoch for comparison
    tib['ts_epoch'] = pd.to_datetime(tib['timestamp']).dt.tz_localize(None)
    # But safer to use trace_window (which has numeric timestamps)
    use_trace_df = trace_window
except Exception:
    use_trace_df = trace_window if 'trace_window' in globals() else (trace_df if 'trace_df' in globals() else pd.DataFrame())

# Ensure trace source exists
if 'use_trace_df' not in locals():
    use_trace_df = trace_df.copy()

# Find earliest trace.self.error_rate per frontend in Window B
trace_src = use_trace_df.copy()
trace_src['timestamp'] = pd.to_numeric(trace_src['timestamp'], errors='coerce')
trace_src = trace_src[(trace_src['timestamp'] >= startB) & (trace_src['timestamp'] <= endB) & (trace_src['trace_name']=='trace.self.error_rate') & (trace_src['cmdb_id'].isin(frontends))]

# For each frontend find earliest timestamp and corresponding value
trace_evidence = []
for fe in frontends:
    sub = trace_src[trace_src['cmdb_id']==fe]
    if not sub.empty:
        idx = sub['timestamp'].idxmin()
        ts_epoch = int(sub.loc[idx,'timestamp'])
        ts_str = pd.to_datetime(ts_epoch, unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z%z')
        val = float(sub.loc[idx,'value'])
    else:
        ts_str = pd.NA
        val = pd.NA
    trace_evidence.append({'cmdb_id':fe,'earliest_trace_self_error_rate_timestamp':ts_str,'trace_self_error_rate_value':val})

trace_evidence_df = pd.DataFrame(trace_evidence)

# 2.b earliest_log_error_count_timestamp and max_error_value from log_df filtered
# Prepare log_df (ensure numeric)
log_df['timestamp'] = pd.to_numeric(log_df['timestamp'], errors='coerce')
log_df['value'] = pd.to_numeric(log_df['value'], errors='coerce')

log_windowB = log_df[(log_df['timestamp'] >= startB) & (log_df['timestamp'] <= endB) & (log_df['log_name']=='log.error_count') & (log_df['cmdb_id'].isin(frontends)) & (log_df['value']>0)].copy()

log_evidence = []
for fe in frontends:
    sub = log_windowB[log_windowB['cmdb_id']==fe]
    if not sub.empty:
        earliest_epoch = int(sub['timestamp'].min())
        earliest_ts = pd.to_datetime(earliest_epoch, unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z%z')
        max_err = float(sub['value'].max())
    else:
        earliest_ts = pd.NA
        max_err = pd.NA
    log_evidence.append({'cmdb_id':fe,'earliest_log_error_count_timestamp':earliest_ts,'max_error_value':max_err})

log_evidence_df = pd.DataFrame(log_evidence)

# 2.c earliest_error_log_timestamp from error_logs_df
error_logs_df['timestamp'] = pd.to_numeric(error_logs_df['timestamp'], errors='coerce')
err_windowB = error_logs_df[(error_logs_df['timestamp'] >= startB) & (error_logs_df['timestamp'] <= endB) & (error_logs_df['cmdb_id'].isin(frontends))].copy()

err_evidence = []
for fe in frontends:
    sub = err_windowB[err_windowB['cmdb_id']==fe]
    if not sub.empty:
        earliest_epoch = int(sub['timestamp'].min())
        earliest_ts = pd.to_datetime(earliest_epoch, unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z%z')
    else:
        earliest_ts = pd.NA
    err_evidence.append({'cmdb_id':fe,'earliest_error_log_timestamp':earliest_ts})

err_evidence_df = pd.DataFrame(err_evidence)

# Merge trace_evidence_df, log_evidence_df, err_evidence_df into frontend_confirmation_B
frontend_confirmation_B = trace_evidence_df.merge(log_evidence_df, on='cmdb_id', how='left').merge(err_evidence_df, on='cmdb_id', how='left')

# Sort by earliest_trace_self_error_rate_timestamp ascending (NaT/NA last)
def parse_sort_val(x):
    try:
        return pd.to_datetime(x)
    except Exception:
        return pd.NaT

frontend_confirmation_B['_sort_ts'] = frontend_confirmation_B['earliest_trace_self_error_rate_timestamp'].apply(parse_sort_val)
frontend_confirmation_B = frontend_confirmation_B.sort_values('_sort_ts', na_position='last').drop(columns=['_sort_ts']).reset_index(drop=True)

# Ensure the requested kernel variable names are set
windowA_component_evidence = windowA_component_evidence
frontend_confirmation_B = frontend_confirmation_B

# Return the two compact tables
windowA_component_evidence, frontend_confirmation_B

```
Out[11]:
```
summary = (
    "Final localization evidence (timestamps in UTC+8):\n\n"
    "Window A (08:30+ candidate — backend/inter-service):\n"
    "- checkoutservice-1: sum_anomaly_count=2637, kpis_with_anomaly=139, earliest=2022-03-20 08:30:00, latest=09:00:00, max_severity≈405.10\n"
    "- checkoutservice-2: sum_anomaly_count=2446, kpis_with_anomaly=131, earliest=2022-03-20 08:30:00, latest=09:00:00, max_severity≈54.43\n"
    "- checkoutservice-0: sum_anomaly_count=2479, kpis_with_anomaly=135, earliest=2022-03-20 08:30:00, latest=09:00:00, max_severity≈10.62\n"
    "- currencyservice-0, shippingservice-0, shippingservice-2: no aggregated metric-anomaly records found in the component-level table (NA) — may lack component-level anomalies in the metric aggregation or were not captured at that aggregation level.\n\n"
    "Interpretation for Window A:\n"
    "- Strong signal of widespread KPI anomalies concentrated on checkoutservice pods beginning at 08:30 and lasting through 09:00, with very high relative severity on some checkout KPIs (checkoutservice-1 notably high).\n"
    "- This supports a backend/inter-service incident affecting the checkout flow (checkout → currency → shipping), although currency/shipping component-level metric aggregates appear NA and should be rechecked at KPI level.\n\n"
    "Window B (08:48–08:52 candidate — frontend-scoped confirmation):\n"
    "- Frontend trace self.error_rate (earliest and value):\n"
    "  * frontend-0 earliest_trace_self_error_rate=2022-03-20 08:48:00, value=0.00813\n"
    "  * frontend-1 earliest_trace_self_error_rate=2022-03-20 08:48:00, value=0.012618\n"
    "  * frontend-2 earliest_trace_self_error_rate=2022-03-20 08:48:00, value=0.00578\n"
    "  * frontend2-0 earliest_trace_self_error_rate=2022-03-20 08:48:00, value=0.008929 (trace record exists)\n"
    "- Corresponding log.error_count evidence (value>0) in same window:\n"
    "  * frontend-0 earliest_log_error_count=2022-03-20 08:48:00, max_error_value=26.0\n"
    "  * frontend-1 earliest_log_error_count=2022-03-20 08:48:00, max_error_value=23.0\n"
    "  * frontend-2 earliest_log_error_count=2022-03-20 08:48:00, max_error_value=24.0\n"
    "  * frontend2-0: no log.error_count >0 found in the window (NA)\n"
    "- Error log messages (error_logs) earliest timestamps:\n"
    "  * frontend-1 earliest_error_log=2022-03-20 08:48:52\n"
    "  * frontend-2 earliest_error_log=2022-03-20 08:48:53\n"
    "  * frontend-0 earliest_error_log=2022-03-20 08:48:54\n"
    "  * frontend2-0: no error_log entries in the window (NA)\n\n"
    "Interpretation for Window B:\n"
    "- Clear, time-correlated evidence across traces (trace.self.error_rate), log.error_count, and error_logs at ~08:48 for frontend-0/1/2 — confirms a frontend-centered incident starting at ~08:48.\n\n"
    "Overall conclusion and recommended next steps:\n"
    "1) Two related incidents: (A) backend checkout-related anomalies starting ~08:30 (checkoutservice* high anomaly counts and severities) and (B) frontend failures starting ~08:48 (frontends show error_rate, log.error_count, and error logs).\n"
    "2) Investigate checkoutservice internals and its downstream calls (currency, shipping) for Window A — inspect KPI-level metrics and traces to locate bottleneck or overload points.\n"
    "3) For Window B, prioritize frontend pods (frontend-0/1/2): correlate request traces for failed requests (trace IDs), check recent deployments/config changes, and review mesh/network metrics between frontends and downstream services.\n"
    "4) Re-check currency/shipping KPI-level metrics (they were NA at the component-aggregate level) to confirm if they were impacted but not captured at the aggregated view.\n\n"
    "Stored variables for follow-up: windowA_component_evidence, frontend_confirmation_B (component-level evidence), and trace/metric instance tables in kernel for deeper drill-down."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(             cmdb_id sum_anomaly_count number_of_kpis_with_anomaly       earliest_anom_timestamp         latest_anom_timestamp max_severity
0  checkoutservice-0              2479                         135  2022-03-20 08:30:00 CST+0800  2022-03-20 09:00:00 CST+0800    10.619048
1  checkoutservice-1              2637                         139  2022-03-20 08:30:00 CST+0800  2022-03-20 09:00:00 CST+0800   405.097561
2  checkoutservice-2              2446                         131  2022-03-20 08:30:00 CST+0800  2022-03-20 09:00:00 CST+0800    54.431131
3  currencyservice-0              <NA>                        <NA>                          <NA>                          <NA>         <NA>
4  shippingservice-0              <NA>                        <NA>                          <NA>                          <NA>         <NA>
5  shippingservice-2              <NA>                        <NA>                          <NA>                          <NA>         <NA>,        cmdb_id earliest_trace_self_error_rate_timestamp  trace_self_error_rate_value earliest_log_error_count_timestamp max_error_value  earliest_error_log_timestamp
0   frontend-0             2022-03-20 08:48:00 CST+0800                     0.008130       2022-03-20 08:48:00 CST+0800            26.0  2022-03-20 08:48:54 CST+0800
1   frontend-1             2022-03-20 08:48:00 CST+0800                     0.012618       2022-03-20 08:48:00 CST+0800            23.0  2022-03-20 08:48:52 CST+0800
2   frontend-2             2022-03-20 08:48:00 CST+0800                     0.005780       2022-03-20 08:48:00 CST+0800            24.0  2022-03-20 08:48:53 CST+0800
3  frontend2-0             2022-03-20 08:48:00 CST+0800                     0.008929                               <NA>            <NA>                          <NA>)```
```